# Medical Report Generation from Chest X-ray Images with a Small Vision-Language Model

**An individual research project — Ben Brahim Mazen**

**Research question:** *Can a small vision-language model adapted with QLoRA generate chest X-ray reports that are clinically faithful, and how often does it hallucinate findings compared with a zero-shot VLM and simple baselines?*

## AI-usage disclosure
Claude (Anthropic, via Claude Code) was used as a **code assistant**: writing and debugging code, and answering technical questions during development. All project decisions — the research question, the dataset, the models, the approach and the evaluation protocol — were chosen and validated by the author, who also ran every experiment. **All results shown in this notebook come from code executed by the author**, and every choice and result can be explained and defended by the author. For the 50-report hallucination review (Stage 7.1), Claude additionally produced a first-pass suggested classification for each sampled report (the `ai_hint_*` columns in `hallucination_sample.csv`); the author read every sampled report and made each final judgment (`manual_category`), confirming or correcting those suggestions. For the 60-report extractor validation (Stage 2.8), Claude likewise produced a first-pass suggested label for every report and every label (the `ai_hint_*` columns in `extractor_validation_sample.csv`); the author read each report and set every final label (`manual_*`), confirming or correcting those suggestions — the per-label agreement reported in this notebook is computed from the author’s labels only.

# Contents — where to find each expected section

The notebook follows the structure of a research report (Introduction → state of the art → data → methodology → implementation → experimental protocol → results → critical analysis → limitations and perspectives → references). The report sections sit at the top; the **Stages** below them are the annotated implementation.

| Expected section | Where |
|---|---|
| Introduction and problem statement | **§1** (just below) |
| State of the art and positioning | **§2** · references in §8.6 |
| Data and preparation | **§3.1** · Stage 1 (download/inspection) · Stage 2 (cleaning, split, EDA) |
| Methodology (systems, protocol) | **§3.2–3.3** · design-rationale headers of Stages 3–5 |
| Implementation | Stages 1–7 (all code annotated, runs top to bottom) |
| Experimental protocol | **§3.3** · Stages 3.2–3.4 (metrics + bootstrap) · Stages 2.8 / 7.1 (manual protocols) |
| Results | Stage 6 (main table + CIs) · §8.2 |
| Critical analysis | §8.3 · Stage 7 (hallucination rates, cost, qualitative examples) |
| Limitations and perspectives | §8.4 |
| Conclusion | §8.5 |
| References | §8.6 |
| Reproducibility | §3.4 · §8.7 · `VERSION_LOG` (Stage 1.2) |
| AI-usage disclosure | Header cell (top of notebook) |

**Quick-reading path:** §1 → §2.4 (positioning) → §3 → Stage 6 table → §8.2–8.5.

# 1. Introduction and problem statement

## 1.1 Context: the reporting bottleneck
Medical imaging volume keeps growing faster than reporting capacity, and the chest X-ray is the most frequently performed imaging examination worldwide. Every exam must end in a written report, and the report — not the image — is what other clinicians actually read. Writing those reports is repetitive, time-pressured work: chest reports are short, semi-formulaic (a large share are variants of "no acute cardiopulmonary abnormality"), yet the wording carries clinical decisions. That mix — formulaic enough to look automatable, consequential enough to be dangerous to automate badly — is exactly the profile where language models promise a lot and fail subtly.

## 1.2 Why this is hard: clinical faithfulness, not fluency
A report generator can be fluent, plausible, and *wrong in the clinically decisive way*. The two failure modes that matter are **hallucination** (asserting a finding the image does not show — e.g. "small pleural effusion" when there is none) and **omission** (missing a finding the image does show). Both are invisible to standard text-overlap metrics: "no effusion" and "effusion" share almost all their n-grams. So the *evaluation protocol* is as much a part of this project as the model: any claim about clinical faithfulness needs assertion-level metrics and human review, not BLEU alone.

## 1.3 Problem statement and research question
**Can a small vision-language model adapted with QLoRA generate chest X-ray reports that are clinically faithful, and how often does it hallucinate findings compared with a zero-shot VLM and simple baselines?**

This decomposes into four operational questions answered by Stages 3–7:
1. Does task adaptation (QLoRA) improve *clinical-label* agreement over the same model used zero-shot? (B3 vs M)
2. Does any generator beat the trivial-but-strong "always answer normal" strategy of this dataset? (M vs B1)
3. How much of the task is solvable by pure visual similarity, with no generation at all? (B2)
4. What *kinds* of hallucination occur, and at what rate — measured manually, not just by metric deltas? (Stage 7)

## 1.4 Objectives and contributions
- **A complete, reproducible protocol** on open data (IU X-Ray) and budget-capped cloud GPUs (free tiers, then a small pay-per-second credit): patient-level split with leakage assertions, negation-aware clinical-label extraction, bootstrap CIs, manual hallucination taxonomy, cost accounting.
- **A controlled comparison** — template, retrieval, zero-shot VLM, QLoRA r=32, QLoRA r=8 — designed so each gap between systems answers one question (§3.2).
- **An honest analysis**: overlap metrics vs clinical metrics contrasted; every number computed from code in this notebook; negative results reported, not hidden.

## 1.5 What this project is not
This is a research exercise on public, de-identified data — **not a clinical decision tool**. Nothing here is validated for patient care; the interest is methodological: how to *measure* whether a generated radiology report is clinically faithful, and what a realistically budgeted adaptation recipe actually buys.

## 1.6 Notebook roadmap
§2 state of the art → §3 methodology overview → **Stage 1** setup/data → **Stage 2** cleaning, patient split, extractor → **Stage 3** metrics + baselines B1/B2 → **Stage 4** VLM load, B3, smoke test → **Stage 5** QLoRA training + ablation → **Stage 6** main results with CIs → **Stage 7** hallucination/cost/qualitative analysis → **Stage 8 (§8)** results discussion, limitations, references.

# 2. State of the art and positioning

## 2.1 Radiology report generation: the medical-captioning line of work
The task was first framed as medical image captioning at the study level: an encoder-decoder network produces one paragraph of findings for a set of images. **R2Gen** (Chen et al., 2020) established the now-standard setup — IU X-Ray as benchmark, relational memory in the decoder, BLEU/ROUGE/METEOR as metrics — and a long line of follow-ups (relational graphs, cross-modal memory, contrastive objectives) refined that recipe. Those systems are trained *from scratch on ~3k reports*, which bounds how much language knowledge they can acquire; our working hypothesis is that a modern instruction-tuned VLM already holds that knowledge and needs *adaptation*, not learning from zero.

## 2.2 The clinical-evaluation problem
Overlap metrics compare generated and reference *strings*. Clinical faithfulness compares *assertions*: "no large effusion" and "large effusion" are near-identical n-grams and opposite findings. **CheXpert** (Irvin et al., 2019) introduced the tools that address this: a labeler that reads a report and returns, per finding, *positive / uncertain / negative / unmentioned*, handling negation ("no evidence of"), uncertainty ("cannot exclude") and hedged phrasing. That philosophy underlies label-based evaluation of generated reports (precision/recall per finding rather than n-gram overlap). Our Stage 2.6 extractor is a small, fully transparent reimplementation of this idea over 7 findings — and, unlike a black-box labeler, its own error rate is measured on a manually labelled sample (Stage 2.8) rather than assumed. *(Validation status: the 60-report stratified sample is drawn and exported in Stage 2.8; the agreement figure is quoted here once the author's manual labelling pass is complete.)*

## 2.3 Open VLMs and parameter-efficient adaptation
General-purpose vision-language models became instruction-following multi-image readers: **Qwen2.5-VL** (Bai et al., 2025) is an open-weights family whose 3B variant reads document-style images at controllable resolution and performs strongly on general vision-language benchmarks — but, being general-purpose, it has no notion of radiology report style, terminology, or the negation-heavy phrasing of FINDINGS sections. Medical variants exist (e.g. **MedGemma**, Sellergren et al., 2025) but are larger or access-gated. **LoRA** (Hu et al., 2021) freezes the base model and trains low-rank adapters; **QLoRA** (Dettmers et al., 2023) combines 4-bit NF4 quantisation of the frozen base with adapter training, bringing 3–4B-model fine-tuning within reach of a single free 16 GB GPU. Whether that recipe improves *clinical faithfulness* on ~2.5k training reports — not merely text overlap — is exactly our research question.

## 2.4 Positioning of this project
This project does not propose a new architecture. Its contribution is a **rigorous, fully reproducible comparison** that prior work on IU X-Ray typically skips: (i) zero-shot vs QLoRA-adapted *same* VLM, isolating what adaptation buys; (ii) *sane* baselines — majority-template and CLIP retrieval — that quantify how far pattern-matching alone gets you on a normal-dominated dataset; (iii) evaluation centred on clinical labels with bootstrap CIs **plus a manual hallucination taxonomy**, not BLEU alone; (iv) an ablation on adapter rank; (v) total cost accounting (time, memory, money = 0 €) under an explicit free-GPU constraint. An honest negative result — if QLoRA mainly learns the *style* of IU X-Ray reports while hallucination stays comparable — is itself an informative finding for the "small VLM + QLoRA" recipe at this data scale.

# 3. Methodology — overview

*(Each item points to the stage where it is implemented and printed.)*

## 3.1 Data
**IU X-Ray / Open-I** (Demner-Fushman et al.): open access (no credential wait, unlike MIMIC-CXR), the standard small benchmark for the task — ~3–4k studies, frontal + lateral chest radiographs, FINDINGS/IMPRESSION text. Target = FINDINGS, falling back to IMPRESSION; cleaning deliberately minimal to preserve clinical signal (negations, laterality). **Patient-level 70/10/20 split** with hard leakage assertions at run time (Stage 2.4): frontal/lateral views of one study are near-duplicates, so an image-level split would inflate scores. Reports with no text or no linked image are excluded and counted (Stage 2.3).

## 3.2 The five compared systems

| System | What it does | Why it is in the comparison |
|---|---|---|
| **B1 template** | Always outputs the most frequent normal train report | The deceptively strong bar on a normal-dominated dataset; a generator must beat it to be useful |
| **B2 retrieval** | Returns the report of the visually nearest train study (CLIP embeddings) | Bounds what pure image similarity achieves without any generation |
| **B3 zero-shot VLM** | Qwen2.5-VL-3B, fixed documented prompt, no training | Measures the off-the-shelf ability the model already has |
| **M QLoRA r=32** | B3's model adapted to the task (our main contribution) | The research question: does small-scale adaptation buy clinical faithfulness? |
| **Ablation r=8** | Same as M with lower adapter rank | One clean knob: does adapter capacity matter at this data scale? |

## 3.3 Evaluation protocol
- **Primary — clinical faithfulness:** per-label precision/recall/F1 + macro-F1 over 7 findings, extracted from generated and reference texts by the **same transparent, negation-aware rule extractor** (Stage 2.6), whose error rate is itself measured on a 60-report manual validation sample (Stage 2.8).
- **Secondary — text overlap:** BLEU-4, ROUGE-L, BERTScore, reported because the literature reports them, and contrasted with the clinical metrics (the expected disagreement is a key finding).
- **Uncertainty:** bootstrap 95% CIs on BLEU-4, ROUGE-L and clinical macro-F1 (Stage 3.4).
- **Error analysis:** manual review of 50 reports with a fixed hallucination taxonomy (false-positive finding, missed finding, wrong laterality/location, contradiction, fluent-but-ungrounded) — Stage 7.
- **Cost analysis:** trainable parameters, training time, peak memory, per-report latency — Stage 7.3 (required by the brief).
- All generative systems use greedy decoding and the same prompt at train and inference time; every generation is persisted so metrics are recomputable independently.

## 3.4 Compute and reproducibility
Free-tier GPUs for as long as they lasted: Google Colab T4 (baselines, metrics) and Kaggle T4 ×2 (the 4-bit model sharded across the two GPUs with `device_map='auto'`) — each was ultimately lost to quota revocations or session time caps, so the fine-tuning runs moved to a rented pay-per-second GPU (Modal, Nvidia A10G 24 GB, ≈ $1.10/h) under a $30 platform credit; the total spend is reported with the results (Stage 7.3 / §8.3). 4-bit NF4 quantised base model + LoRA adapters only, gradient checkpointing, batch 1 × gradient accumulation 8. Seeds fixed (`SEED=42`), library versions and GPU logged (`VERSION_LOG`), every artefact (generations, adapters, logs, tables) persisted to Drive (Colab), notebook output (Kaggle) or a persistent Volume (Modal). Full statement in §8.7.

# STAGE 1 — Environment setup, version logging, seeds, dataset download and first inspection

**How to run — Modal (the fine-tuning runs):** `modal launch jupyter --gpu A10G --cpu 4 --memory 16384 --timeout 86400 --volume <volume> --mount <folder-with-this-notebook> --detach`, open the printed URL, then Run All. Prior artefacts are pre-uploaded into the volume (`modal volume put <volume> <folder> /import_pack`); Stage 1.3 imports them from `/root/lab/volume`, and the volume's background commits keep every later write durable while the session runs.
**How to run — Google Colab:** `Runtime → Change runtime type → T4 GPU` → `Runtime → Run all`; authorize the Google Drive popup within the first minute.
**How to run — Kaggle:** notebook Settings → Accelerator `GPU T4 ×2`, Internet `ON`, prior artefacts attached as an Input → `Run All`. Stage 1.3 imports those artefacts from `/kaggle/input` automatically (files, folders, or a single zip all work).

**The notebook is one self-contained Run-All pipeline.** Every stage is idempotent: work already persisted (generations, adapter train logs, results-table rows, CLIP cache, manually filled CSVs) is detected and skipped with a `[skip]` line, and interrupted generation passes resume from their partial saves. The single execution switch is `RUN_ABLATION` in Stage 5.3 (default `True`: one session trains m_r32 **and** m_r8 back to back; set `False` to train the main model only). A fresh session needs ~15–25 min of setup before training starts (installs, the ~1.4 GB image archive, model download).

In [ ]:
# =====================================================================
# STAGE 1.1 — Installations
# Minimum versions are forced where a feature depends on them:
#   transformers >= 4.49 : adds the `qwen2_5_vl` architecture (Qwen2.5-VL)
#   transformers >= 4.50 : Gemma 3 support (only needed for the MedGemma fallback)
#   transformers <  5    : pin out a hypothetical future major rewrite
#                         (class names / APIs may move — protect the run)
# Colab/Kaggle images preinstall torch/pandas/matplotlib; a bare cloud
# image (Modal) does not, so the full scientific stack is installed
# there explicitly — torch FIRST so the other packages see it and pip
# does not drag a second copy in through their dependencies.
# =====================================================================
import os as _os
if _os.path.isdir('/root/lab/volume'):            # Modal: bare Ubuntu image
    !pip install -q torch torchvision ipywidgets pandas matplotlib pillow tqdm requests
!pip install -q -U 'transformers>=4.50.0,<5.0.0' accelerate peft bitsandbytes
!pip install -q -U qwen-vl-utils evaluate rouge_score bert-score nltk sentencepiece scikit-learn


In [ ]:
# =====================================================================
# STAGE 1.2 — Imports, seeds, GPU check, version logging
# =====================================================================
import os, sys, random, platform, datetime
import numpy as np
import pandas as pd
import torch

SEED = 42

def set_all_seeds(seed=SEED):
    '''Fix every random generator this project will touch.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False    # deterministic conv algorithms
    torch.backends.cudnn.deterministic = True

set_all_seeds()

import transformers, accelerate, peft, bitsandbytes, sklearn
from tqdm.notebook import tqdm

def _major_minor(v):
    parts = v.split('.')
    return (int(parts[0]), int(parts[1]))

assert torch.cuda.is_available(), 'No GPU detected. Colab: Runtime > Change runtime type > T4 GPU.'
assert _major_minor(transformers.__version__) >= (4, 50), (
    f'transformers {transformers.__version__} is too old for Qwen2.5-VL / Gemma 3. Re-run the install cell.')

VERSION_LOG = {
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'python':        platform.python_version(),
    'torch':         torch.__version__,
    'transformers':  transformers.__version__,
    'accelerate':    accelerate.__version__,
    'peft':          peft.__version__,
    'bitsandbytes':  bitsandbytes.__version__,
    'scikit-learn':  sklearn.__version__,
    'pandas':        pd.__version__,
    'numpy':         np.__version__,
    'cuda':          torch.version.cuda,
    'gpu_name':      torch.cuda.get_device_name(0),
    'gpu_total_gb':  round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
}
VERSION_LOG


In [ ]:
# =====================================================================
# STAGE 1.3 — Workspace: session disk for data, persistent dir for artefacts
# Platform map:
#   Colab : LOCAL on the VM disk, PERSIST on mounted Google Drive
#   Kaggle: LOCAL/PERSIST in /kaggle/working; prior artefacts arrive as a
#           notebook Input under /kaggle/input
#   Modal : LOCAL on the container disk, PERSIST inside a persistent
#           Volume mounted at /root/lab/volume (set up by
#           `modal launch jupyter --volume ...`); prior artefacts are
#           pre-uploaded into that volume, and the volume's background
#           commits make every later write durable while the session runs
# Uploads can bury the artefacts under wrappers of ANY name and depth
# (username folders, zip roots, a zip nested inside a username folder,
# an 'import_pack' folder, ...), so instead of guessing the layout we
# walk the whole input tree, extract every zip found, and pull each
# known artefact file/folder up to PERSIST root — where every
# skip-guard and resume logic below expects it. The bulky session dir
# 'iu_xray' is skipped (re-downloaded anyway).
# =====================================================================
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
IN_MODAL  = os.path.isdir('/root/lab/volume')

if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    LOCAL, PERSIST = '/content/iu_xray', '/content/drive/MyDrive/projet_llm_iu_xray'
else:
    # Kaggle and Modal share the same recursive artefact import
    import shutil as _shutil
    import zipfile as _zipfile
    if IN_KAGGLE:
        LOCAL         = '/kaggle/working/iu_xray'
        PERSIST       = '/kaggle/working/projet_llm_iu_xray'
        _IMPORT_TREES = ['/kaggle/input']
        _ZIP_SCRATCH  = '/kaggle/working/_import_zips'
    else:                                            # Modal
        LOCAL         = '/root/iu_xray'
        PERSIST       = '/root/lab/volume/projet_llm_iu_xray'
        _IMPORT_TREES = ['/root/lab/volume']         # artefacts pre-uploaded here
        _ZIP_SCRATCH  = '/root/_import_zips'

    _ARTEFACT_FILES = ('results_table.csv', 'reports_stage2.csv',
                       'reports_raw_stage1.csv', 'labels_stage2.csv',
                       'clip_image_embeddings.npz',
                       'extractor_validation_sample.csv',
                       'main_results_table.csv', 'cost_table.csv',
                       'hallucination_sample.csv', 'hallucination_rates.csv')
    _ARTEFACT_DIRS  = ('generations', 'adapters')
    _PRUNE_DIRS     = ('iu_xray', '_import_zips')    # session data / scratch

    def _is_marker(name, is_dir):
        return (name.startswith('train_log')                 # train_log_<tag>.json
                or (is_dir and name in _ARTEFACT_DIRS)
                or (not is_dir and name in _ARTEFACT_FILES))

    def _pull_artefacts(tree, label):
        '''Copy every known artefact found ANYWHERE under `tree` — however
        deep the upload wrappers bury it — up to PERSIST root. First copy
        wins, so re-runs are no-ops and nothing genuine is overwritten.'''
        _found = 0
        for _root, _dirs, _files in os.walk(tree):
            _dirs[:] = sorted(d for d in _dirs if d not in _PRUNE_DIRS)
            for _name in list(_dirs) + sorted(_files):
                if not _is_marker(_name, _name in _dirs):
                    continue
                _src = os.path.join(_root, _name)
                _dst = os.path.join(PERSIST, _name)
                if os.path.exists(_dst):
                    continue
                if os.path.isdir(_src):
                    _shutil.copytree(_src, _dst)
                else:
                    _shutil.copy2(_src, _dst)
                _found += 1
                print(f'[import] {label}: {_name}  <- '
                      f'{os.path.relpath(_src, tree)}')
        if not _found:
            print(f'[import] {label}: no artefacts found')

    # 1) extract every zip in the input trees (any depth) into a scratch dir
    os.makedirs(PERSIST, exist_ok=True)
    os.makedirs(_ZIP_SCRATCH, exist_ok=True)
    for _tree in _IMPORT_TREES:
        for _root, _dirs, _files in os.walk(_tree):
            _dirs[:] = sorted(d for d in _dirs if d not in _PRUNE_DIRS)
            for _fn in sorted(f for f in _files if f.endswith('.zip')):
                _zdst = os.path.join(_ZIP_SCRATCH, _fn[:-4])
                if os.path.exists(_zdst):
                    continue                              # already extracted
                print(f'[import] extracting {_fn}')
                with _zipfile.ZipFile(os.path.join(_root, _fn)) as _z:
                    _z.extractall(_zdst)

    # 2) pull artefacts from the input trees and from the extracted zips
    for _tree in _IMPORT_TREES:
        _label = os.path.basename(_tree.rstrip('/')) or 'input'
        _pull_artefacts(_tree, _label)
    _pull_artefacts(_ZIP_SCRATCH, 'zip')
    print('[import] PERSIST now contains:', sorted(os.listdir(PERSIST)))

RAW = os.path.join(LOCAL, 'raw')
for d in (os.path.join(RAW, 'images'), os.path.join(RAW, 'reports'), PERSIST):
    os.makedirs(d, exist_ok=True)
print('platform          :',
      'modal' if IN_MODAL else ('kaggle' if IN_KAGGLE else ('colab' if IN_COLAB else 'local')))
print('LOCAL session dir :', LOCAL)
print('Persistent dir    :', PERSIST)


In [ ]:
# =====================================================================
# STAGE 1.4 — Download IU X-Ray (Open-I / NLMCXR collection)
# Official source (US National Library of Medicine, Open-i):
#   images : https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz
#   reports: https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz
# The NLM endpoint intermittently serves an HTML error page WITH HTTP 200
# (observed 2026-10-04), so every file is validated after download (gzip
# magic bytes + minimum size), an already-present invalid file is deleted
# and re-fetched, and each archive falls back to an Internet Archive
# snapshot of the SAME official URL (verified: identical official files,
# 3,955 XML reports / 7,470 PNGs).
# =====================================================================
import requests

_UA = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
                     '(KHTML, like Gecko) Chrome/130.0 Safari/537.36'}
_MIN_TGZ_BYTES = 1_000_000        # both archives are far larger; an HTML page is ~14 KB

SOURCES = {
    'images': [
        'https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz',
        'https://web.archive.org/web/20250610002550/https://openi.nlm.nih.gov/imgs/collections/NLMCXR_png.tgz',
    ],
    'reports': [
        'https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz',
        'https://web.archive.org/web/20250526091239/https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz',
    ],
}

def _looks_like_tgz(path):
    try:
        with open(path, 'rb') as f:
            return f.read(2) == b'\x1f\x8b' and os.path.getsize(path) > _MIN_TGZ_BYTES
    except OSError:
        return False

def download_file(urls, dest):
    if os.path.exists(dest):
        if _looks_like_tgz(dest):
            print(f'[skip] {os.path.basename(dest)} already present')
            return
        print(f'[bad ] {os.path.basename(dest)} is not a valid archive — deleted, re-fetching')
        os.remove(dest)
    errors = []
    for _attempt in range(2):                      # one full retry round
        for url in urls:
            try:
                print(f'[get ] {url}')
                with requests.get(url, stream=True, timeout=120, headers=_UA) as r:
                    r.raise_for_status()
                    total = int(r.headers.get('content-length', 0))
                    if total and total < _MIN_TGZ_BYTES:
                        raise IOError(f'only {total} bytes served — not the real archive')
                    with open(dest + '.part', 'wb') as f, tqdm(total=total or None, unit='B',
                                                               unit_scale=True, desc=os.path.basename(dest)) as bar:
                        for chunk in r.iter_content(chunk_size=1 << 20):
                            f.write(chunk)
                            bar.update(len(chunk))
                if not _looks_like_tgz(dest + '.part'):
                    raise IOError('downloaded bytes are not a gzip archive (HTML error page?)')
                os.replace(dest + '.part', dest)
                print(f'[done] {dest} ({os.path.getsize(dest)/1e6:.1f} MB)')
                return
            except Exception as e:
                errors.append(f'{url}: {e}')
                print(f'[fail] {e}')
                if os.path.exists(dest + '.part'):
                    os.remove(dest + '.part')
    raise RuntimeError('all download sources failed:\n  ' + '\n  '.join(errors))

for name, urls in SOURCES.items():
    download_file(urls, os.path.join(RAW, name + '.tgz'))

# LAST-RESORT FALLBACK — only if BOTH the NLM endpoint and the Internet
# Archive are unreachable. Kaggle mirror raddar/chest-xrays-indiana-
# university has the PNG images but CSVs instead of XML reports, so the
# Stage 2 XML parsing would need changes: treat as break-glass only.

In [ ]:
# =====================================================================
# STAGE 1.5 — Extraction + inventory (file counts are PRINTED, never assumed)
# =====================================================================
import glob, tarfile

def safe_extract(tgz_path, dest):
    marker = os.path.join(dest, '.extracted')
    if os.path.exists(marker):
        print(f'[skip] {os.path.basename(tgz_path)} already extracted')
        return
    os.makedirs(dest, exist_ok=True)
    print(f'[open] {tgz_path}')
    with tarfile.open(tgz_path) as tar:
        try:
            tar.extractall(dest, filter='data')   # safe on Python >= 3.12
        except TypeError:
            tar.extractall(dest)                  # older Pythons
    open(marker, 'w').close()
    print(f'[done] extracted to {dest}')

safe_extract(os.path.join(RAW, 'images.tgz'),  os.path.join(RAW, 'images'))
safe_extract(os.path.join(RAW, 'reports.tgz'), os.path.join(RAW, 'reports'))

PNG_FILES = sorted(glob.glob(os.path.join(RAW, 'images', '**', '*.png'), recursive=True))
XML_FILES = sorted(glob.glob(os.path.join(RAW, 'reports', '**', '*.xml'), recursive=True))
print(f'PNG images found : {len(PNG_FILES)}')
print(f'XML reports found: {len(XML_FILES)}')
print('example image :', os.path.basename(PNG_FILES[0]) if PNG_FILES else 'NONE')
print('example report:', os.path.basename(XML_FILES[0]) if XML_FILES else 'NONE')


In [ ]:
# =====================================================================
# STAGE 1.6 — Ground truth before parsing: look at one raw XML report
# =====================================================================
from pathlib import Path

first_xml = XML_FILES[0]
raw_text = Path(first_xml).read_text(encoding='utf-8', errors='replace')
print(f'--- {os.path.basename(first_xml)} (first 2000 chars) ---')
print(raw_text[:2000])


In [ ]:
# =====================================================================
# STAGE 1.7 — Minimal parser + view one image next to its report
# =====================================================================
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from PIL import Image
import re as _re

def parse_report(xml_path):
    '''Extract the raw structure of one Open-I report (no cleaning yet).'''
    root = ET.parse(xml_path).getroot()
    sections = {a.get('Label'): (a.text or '').strip() for a in root.iter('AbstractText')}
    image_ids = [p.get('id') for p in root.iter('parentImage')]
    return {'report_file': os.path.basename(xml_path),
            'sections': sections,
            'image_ids': image_ids}

def find_png(image_id):
    '''XML ids carry an extra "_<k>" segment ("CXR1_1_IM-0001-3001") that
    the PNG filename lacks ("CXR1_IM-0001-3001.png") — strip it, then
    match exactly. Ids without the extra segment pass through unchanged.'''
    stem = _re.sub(r'^(CXR\d+)_\d+(_IM-\d+-\d+)$', r'\1\2', image_id)
    hits = [p for p in PNG_FILES if os.path.basename(p) == stem + '.png']
    return hits[0] if hits else None

rep = parse_report(XML_FILES[0])
print('sections found :', list(rep['sections'].keys()))
print('image ids      :', rep['image_ids'])

png_path = find_png(rep['image_ids'][0]) if rep['image_ids'] else None
if png_path:
    img = Image.open(png_path)
    print(f'image: {os.path.basename(png_path)}  size={img.size}  mode={img.mode}')
    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap='gray')
    plt.axis('off'); plt.title(os.path.basename(png_path)); plt.show()
else:
    print('!! no PNG matched the first parentImage id — inspect naming')

for label, txt in rep['sections'].items():
    print(f'\n[{label}]\n{txt[:400] if txt else "(empty)"}')


In [ ]:
# =====================================================================
# STAGE 1.8 — Raw dataframe of ALL reports (Stage 2 cleans + splits it)
# =====================================================================
records = []
for p in tqdm(XML_FILES, desc='parsing reports'):
    r = parse_report(p)
    records.append({
        'report_file': r['report_file'],
        'image_ids': '|'.join(r['image_ids']),
        'n_images': len(r['image_ids']),
        **{f'sec_{k.lower()}': v for k, v in r['sections'].items()},
    })

df_reports = pd.DataFrame(records)
df_reports.to_csv(os.path.join(RAW, 'reports_raw.csv'), index=False)
df_reports.to_csv(os.path.join(PERSIST, 'reports_raw_stage1.csv'), index=False)  # backup
print('shape   :', df_reports.shape)
print('columns :', df_reports.columns.tolist())
df_reports.head()


# STAGE 2 — Cleaning, patient-level split, EDA, and the clinical-label extractor

**What this stage does**
1. Reload the raw reports and **verify the image-id structure** before trusting any grouping.
2. Build the generation target (FINDINGS, falling back to IMPRESSION) with minimal, documented cleaning.
3. **Split by patient (70/10/20)** with hard leakage assertions.
4. EDA: section availability, target length, normal-vs-abnormal share (why baseline B1 matters).
5. A **transparent, negation-aware rule extractor** for 7 clinical labels + a 60-report **manual validation sample** for the author to fill.

Assumptions about id/section structure are *printed and asserted*, never silent — if a later run contradicts them, the cell fails loudly and we adjust.


In [ ]:
# =====================================================================
# STAGE 2.1 — Reload raw reports (uses Stage 1 outputs of this session,
# or the Drive backup CSV if the session is fresh)
# =====================================================================
import re
import matplotlib.pyplot as plt

raw_candidates = [os.path.join(RAW, 'reports_raw.csv'),
                  os.path.join(PERSIST, 'reports_raw_stage1.csv')]
raw_csv = next((p for p in raw_candidates if os.path.exists(p)), None)
assert raw_csv is not None, 'reports_raw.csv not found — run Stage 1 first.'
df = pd.read_csv(raw_csv)
print('loaded :', raw_csv)
print('shape  :', df.shape)
print('columns:', df.columns.tolist())


In [ ]:
# =====================================================================
# STAGE 2.2 — Image-id structure + PATIENT key
# IU X-Ray ids typically look like 'CXR108_IM-0177-1001' (VERIFY against
# the printed sample): CXR number = patient/visit, IM number = image,
# trailing number = study. We split by the BROADER CXR number so that
# frontal+lateral views of a study — and any repeat studies of one
# patient — always stay on the same side of the split.
# =====================================================================
df_exp = df.assign(image_id=df['image_ids'].str.split('|')).explode('image_id')
df_exp['image_id'] = df_exp['image_id'].fillna('').str.strip()
df_exp = df_exp[df_exp['image_id'] != '']

print('sample image ids:')
print(df_exp['image_id'].head(10).tolist())

pat_re = re.compile(r'CXR(\d+)', flags=re.IGNORECASE)
df_exp['patient_key'] = df_exp['image_id'].map(lambda s: 'CXR_' + pat_re.search(s).group(1)
                                               if pat_re.search(s) else None)
n_bad = df_exp['patient_key'].isna().sum()
print('ids not matching the CXR<digits> pattern:', n_bad)
assert n_bad == 0, 'unexpected id format — paste the sample ids back to Claude'

per_report_pat = df_exp.groupby('report_file')['patient_key'].nunique()
print('reports mapping to more than one patient key:', int((per_report_pat > 1).sum()))
assert (per_report_pat <= 1).all(), 'one report spans several patients — grouping bug'

print(f"reports: {df['report_file'].nunique()} | "
      f"distinct images: {df_exp['image_id'].nunique()} | "
      f"patient keys: {df_exp['patient_key'].nunique()}")

reports_per_patient = df_exp.drop_duplicates(['report_file', 'patient_key']) \
                            .groupby('patient_key').size()
print('\nreports per patient key — describe:')
print(reports_per_patient.describe())
ax = reports_per_patient.value_counts().sort_index().plot(kind='bar', figsize=(6, 3),
                                                           title='reports per patient key')
ax.set_xlabel('reports for one patient'); ax.set_ylabel('number of patients')
plt.show()


In [ ]:
# =====================================================================
# STAGE 2.3 — Section availability, minimal cleaning, TARGET text
# Decisions (defensible in the methodology section):
#   TARGET = FINDINGS; if empty, fall back to IMPRESSION; if both empty
#   the report is dropped (counted). Cleaning is deliberately minimal:
#   radiology findings are short and formulaic, and aggressive
#   normalisation would erase clinical signal (laterality, negations).
# Reports with NO image id are excluded here (counted): they cannot be
# part of an image->report task, and their patient key is derived FROM
# the image id, so keeping them would break the patient-level split.
# =====================================================================
def _col(frame, name):
    return frame[name] if name in frame.columns else pd.Series([''] * len(frame), index=frame.index)

df['findings']    = _col(df, 'sec_findings').fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
df['impression']  = _col(df, 'sec_impression').fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

has_f = df['findings'].str.contains(r'[A-Za-z0-9]')     # '.'-only counts as empty
has_i = df['impression'].str.contains(r'[A-Za-z0-9]')
print(f'missing/empty FINDINGS   : {int((~has_f).sum())} / {len(df)}')
print(f'missing/empty IMPRESSION : {int((~has_i).sum())} / {len(df)}')
print(f'dropped (neither section): {int((~has_f & ~has_i).sum())}')

df['target_text'] = np.where(has_f, df['findings'], df['impression'])
df = df[has_f | has_i].reset_index(drop=True)

img_reports = set(df_exp['report_file'])
n_no_img = int((~df['report_file'].isin(img_reports)).sum())
print(f'reports with no image id (excluded before split): {n_no_img}')
df = df[df['report_file'].isin(img_reports)].reset_index(drop=True)

df['n_words'] = df['target_text'].str.split().str.len()
print('\ntarget length in words — describe:')
print(df['n_words'].describe())
ax = df['n_words'].plot(kind='hist', bins=40, figsize=(6, 3), title='target text length (words)')
ax.set_xlabel('words'); plt.show()

In [ ]:
# =====================================================================
# STAGE 2.4 — Patient-level split (70/10/20) + HARD leakage assertions
# Splitting by image or report would leak: frontal/lateral views of one
# study are near-duplicates, and some patients have several reports.
# =====================================================================
rep2pat = df_exp.drop_duplicates('report_file').set_index('report_file')['patient_key']
df['patient_key'] = df['report_file'].map(rep2pat)
assert df['patient_key'].isna().sum() == 0, 'some reports lost their patient key'

rng = np.random.RandomState(SEED)
patients = np.array(sorted(df['patient_key'].unique()))
rng.shuffle(patients)
n = len(patients)
n_train, n_val = int(0.70 * n), int(0.10 * n)
train_p = set(patients[:n_train])
val_p   = set(patients[n_train:n_train + n_val])
test_p  = set(patients[n_train + n_val:])

df['split'] = np.where(df['patient_key'].isin(train_p), 'train',
              np.where(df['patient_key'].isin(val_p),   'val', 'test'))

# --- leakage assertions (the cell FAILS if data ever crosses splits) ---
assert train_p.isdisjoint(val_p) and train_p.isdisjoint(test_p) and val_p.isdisjoint(test_p)
assert (df.groupby('patient_key')['split'].nunique() == 1).all(), 'LEAKAGE: patient in 2 splits'
print('LEAKAGE CHECK PASSED: every patient key sits in exactly one split')

print('\nreports per split:');  print(df['split'].value_counts())
print('patients per split:', {s: len(set(df.loc[df['split'] == s, 'patient_key']))
                               for s in ('train', 'val', 'test')})
df.to_csv(os.path.join(PERSIST, 'reports_stage2.csv'), index=False)
print("\nsaved -> PERSIST/reports_stage2.csv")


In [ ]:
# =====================================================================
# STAGE 2.5 — EDA: normal-vs-abnormal share (motivates baseline B1)
# Note: this phrase flag is a proxy; the real labels come from the
# extractor + manual validation (2.6-2.8).
# =====================================================================
NORMAL_RE = re.compile(r'\bno acute\b|\bno active disease\b|\bunremarkable\b|'
                       r'\bno (?:cardiopulmonary )?abnormalit\w*\b|\bwithout acute\b',
                       re.IGNORECASE)

df['flag_normal_phrase'] = df['target_text'].str.contains(NORMAL_RE)
share = df.groupby('split')['flag_normal_phrase'].mean().sort_values(ascending=False)
print('share of reports containing a normal / no-acute phrase:')
print((share * 100).round(1).astype(str) + ' %')
ax = share.plot(kind='bar', figsize=(5, 3), title='normal-phrase share per split')
ax.set_ylabel('fraction of reports'); ax.set_xlabel('split'); plt.show()
print('Interpretation: if normal reports dominate, a constant "normal chest"')
print('template (baseline B1) is a deceptively strong competitor — beating it')
print('is what makes a generator actually useful.')


In [ ]:
# =====================================================================
# STAGE 2.6 — Clinical finding extractor (rule-based, negation-aware)
# Same extractor will label REFERENCE and GENERATED reports, so it must
# be transparent (every rule visible here), negation-aware, and its
# errors measured by hand (2.8).
#
# Design (key decisions):
#   1. lowercase, split text into clauses at [,.;:] and ' but ';
#   2. a finding MENTION matches one of its synonym regexes;
#   3. inside its clause, look BEFORE the mention: uncertainty cue
#      -> 'uncertain'; else negation cue -> 'negated'; else 'positive';
#   4. report-level status: any positive > any uncertain > all negated.
# This handles 'no effusion or pneumothorax' (both negated) but not
# deeply nested scope — a limitation we quantify in validation.
# =====================================================================
FINDING_PATTERNS = {
    'cardiomegaly':      [r'cardiomegal\w*', r'enlarged cardiac silhouette',
                          r'cardiac enlargement', r'enlarged heart', r'heart size is enlarged'],
    'pleural_effusion':  [r'pleural effusion', r'effusions?',
                          r'fluid in (?:the )?pleural space', r'hemothorax'],
    'pneumonia_opacity': [r'pneumoni\w*', r'opacit\w*', r'consolidat\w*',
                          r'infiltrat\w*', r'airspace disease'],
    'pneumothorax':      [r'pneumothorax\w*', r'collapsed lung', r'lung collapse'],
    'edema':             [r'\bedema\b', r'oedema', r'pulmonary (?:vascular )?congestion',
                          r'congestive heart failure'],
    'atelectasis':       [r'atelectas\w*'],
}
NEG_CUES = [r'\bno\b', r'\bnot\b', r'\bwithout\b', r'\bfree of\b', r'\babsen(?:ce|t)\b',
            r'\bnegative for\b', r'\brules? out\b', r'\bresolved\b', r'\bresolution\b',
            r'\bdenies\b']
UNC_CUES = [r'\bcannot (?:be )?exclud\w*\b', r'\bcan not exclud\w*\b', r'\bpossible\b',
            r'\bquestionable\b', r'\bmay represent\b', r'\bsuspect\w*\b', r'\bconcern(?:ing)? for\b']

CLAUSE_SPLIT = re.compile(r'[,.;:]|\bbut\b', flags=re.IGNORECASE)
NEG_CUES_C = [re.compile(p) for p in NEG_CUES]
UNC_CUES_C = [re.compile(p) for p in UNC_CUES]

def _clause_status(clause, pos):
    prefix = clause[:pos]
    if any(c.search(prefix) for c in UNC_CUES_C):
        return 'uncertain'
    if any(c.search(prefix) for c in NEG_CUES_C):
        return 'negated'
    return 'positive'

def label_text(text):
    '''Return ({finding: status}, {finding: [(clause_i, match_text, status)]}).'''
    clauses = CLAUSE_SPLIT.split(text.lower())
    result, evidence = {}, {}
    for finding, patterns in FINDING_PATTERNS.items():
        statuses, ev = [], []
        for ci, clause in enumerate(clauses):
            for pat in patterns:
                for m in re.finditer(pat, clause):
                    st = _clause_status(clause, m.start())
                    statuses.append(st)
                    ev.append((ci, m.group(0), st))
        if statuses:
            if 'positive' in statuses:
                s = 'positive'
            elif 'uncertain' in statuses:
                s = 'uncertain'
            else:
                s = 'negated'
            result[finding], evidence[finding] = s, ev
    return result, evidence

def label_report_full(text):
    '''label_text + a derived no_acute_finding label from normal phrases.'''
    d, ev = label_text(text)
    pos_any = any(s == 'positive' for s in d.values())
    if NORMAL_RE.search(text):
        d['no_acute_finding'] = 'uncertain' if pos_any else 'positive'
    else:
        d['no_acute_finding'] = 'unmentioned'
    return d, ev

# --- quick unit-style sanity checks (visible reasoning, not hidden tests) ---
_checks = [
    ('no pleural effusion or pneumothorax', 'pleural_effusion', 'negated'),
    ('no pleural effusion or pneumothorax', 'pneumothorax',    'negated'),
    ('small left pleural effusion',         'pleural_effusion', 'positive'),
    ('cannot exclude early pneumonia',      'pneumonia_opacity', 'uncertain'),
    ('cardiomegaly is present',             'cardiomegaly',     'positive'),
    ('no cardiomegaly',                     'cardiomegaly',     'negated'),
]
for txt, finding, expected in _checks:
    got = label_text(txt)[0].get(finding, 'unmentioned')
    print(f'{"OK " if got == expected else "FAIL"} | {finding:18s} -> {got:10s} | {txt}')
    assert got == expected, f'extractor check failed on: {txt}'


In [ ]:
# =====================================================================
# STAGE 2.7 — Label every reference report + label frequencies (EDA)
# =====================================================================
status_rows = []
for t in df['target_text']:
    d, _ = label_report_full(t)
    status_rows.append(d)
lab = pd.DataFrame(status_rows).fillna('unmentioned')
bin_lab = lab.eq('positive').astype(int)          # binary view: 1 = asserted present

df_labels = pd.concat([df[['report_file', 'split']].reset_index(drop=True),
                       lab.add_prefix('st_'), bin_lab.add_prefix('bin_')], axis=1)
df_labels.to_csv(os.path.join(PERSIST, 'labels_stage2.csv'), index=False)

freq = bin_lab.mean().sort_values(ascending=False)
print('positive-label frequency across reference reports (all splits):')
print((freq * 100).round(1).astype(str) + ' %')
ax = freq.plot(kind='barh', figsize=(6, 3), title='reference label prevalence')
ax.set_xlabel('fraction of reports'); plt.show()
print('\nuncertain mentions (kept out of the binary view, logged for analysis):')
print((lab == 'uncertain').sum().sort_values(ascending=False))


In [ ]:
# =====================================================================
# STAGE 2.8 — Extractor validation sample (MANUAL STEP for the author)
# 60 reports sampled from train+val ONLY (never test: no peeking).
# Fill every manual_<label> cell with P / N / U by READING the report:
#   P = the finding is clearly asserted as PRESENT
#   N = clearly absent/negated/unmentioned
#   U = you cannot tell (this is exactly what we want to measure)
# Then we compute per-label agreement in the next session.
# GUARD: an existing sample WITH manual labels is kept — a re-run must
# never destroy manual work.
# =====================================================================
_val_p = os.path.join(PERSIST, 'extractor_validation_sample.csv')
_keep = False
if os.path.exists(_val_p):
    _prev = pd.read_csv(_val_p)
    _mc = [c for c in _prev.columns if c.startswith('manual_')]
    _keep = any(_prev[c].fillna('').astype(str).str.strip().ne('').any()
                for c in _mc)

if _keep:
    print('extractor_validation_sample.csv already carries manual labels — kept (no re-sample)')
    export = _prev
else:
    pool = df[df['split'].isin(['train', 'val'])]
    sample = pool.sample(n=60, random_state=SEED)
    lab_idx = df_labels.set_index('report_file')
    export = sample[['report_file', 'split', 'target_text']].copy()
    for f in list(FINDING_PATTERNS) + ['no_acute_finding']:
        export['auto_' + f] = sample['report_file'].map(lab_idx['st_' + f]).fillna('unmentioned')
        export['manual_' + f] = ''
    export.to_csv(_val_p, index=False)
    print('saved -> PERSIST/extractor_validation_sample.csv  (also display below)')
export[['report_file', 'split', 'target_text']].head(5)


## Stage 2 — checks before moving on

- [ ] Sample image ids printed and all matched the `CXR<digits>` pattern (assertion passed).
- [ ] Missing-section counts, dropped-report count, and **excluded imageless-report count** printed; split counts ≈ 70/10/20 of reports.
- [ ] **Leakage assertion passed** (every patient key in exactly one split).
- [ ] Extractor sanity checks all printed `OK`.
- [ ] Label prevalence bar chart printed; normal-phrase share computed.
- [x] `extractor_validation_sample.csv` saved — **author filled all `manual_*` columns (420/420, 2026-10-08); agreement reported in §8.1**.

**Paste back to Claude:** the id sample + patient/report counts, split counts + leakage line, missing/dropped/excluded counts, normal-phrase shares, label frequencies, and the extractor check lines (OK/FAIL).

**Next (Stage 3–4):** baselines B1 (template) and B2 (retrieval), VLM load + B3 zero-shot, 100-sample smoke test, QLoRA configuration.

# STAGE 3 — Evaluation machinery + baselines B1 (template) and B2 (retrieval)

**Why baselines first:** a generative model is only meaningful *relative to something trivial*. IU X-Ray is dominated by near-normal reports, so a constant "normal chest" template (B1) is a deceptively strong competitor — beating it is the actual bar. B2 (return the report of the visually nearest training study) is the non-generative reference: it bounds what pure image similarity achieves.

This stage needs a session where Stage 1 (images extracted) and Stage 2 have run. All metric functions defined here are reused for B3 and the fine-tuned models.


In [ ]:
# =====================================================================
# STAGE 3.1 — Rebuild context: reports, labels, image paths, test order
# THE JOIN (learned from a real run): XML parentImage ids look like
# 'CXR1_1_IM-0001-3001' (extra '_<k>' after the CXR number) while PNG
# filenames are 'CXR1_IM-0001-3001.png'. PNG_MAP therefore indexes every
# image under three keys — exact filename, filename without extension,
# and a normalized key 'CXR<patient>|IM-<img>-<study>' — and lookups
# fall back to the normalized form. The join is verified LOUDLY here
# instead of failing two cells later with a cryptic numpy error.
# =====================================================================
import os, re, json, glob, time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm
import torch

df = pd.read_csv(os.path.join(PERSIST, 'reports_stage2.csv'))
df_labels = pd.read_csv(os.path.join(PERSIST, 'labels_stage2.csv'))
PNG_FILES = sorted(glob.glob(os.path.join(RAW, 'images', '**', '*.png'), recursive=True))
assert len(PNG_FILES) > 0, 'no images in this session — re-run Stage 1 download/extract'

_ID_RE = re.compile(r'CXR(\d+)(?:_\d+)?_IM-(\d+)-(\d+)', re.IGNORECASE)

def _norm_img_key(s):
    '''Normalized image identity: patient number + IM tail. Maps both
    'CXR1_1_IM-0001-3001' (XML id) and 'CXR1_IM-0001-3001.png' (file) to
    the same key, so the join is insensitive to the extra '_<k>' segment
    and to leading zeros.'''
    m = _ID_RE.search(s)
    if not m:
        return None
    return f'CXR{int(m.group(1))}|IM-{int(m.group(2))}-{int(m.group(3))}'

PNG_MAP = {}
for p in PNG_FILES:
    b = os.path.basename(p)
    PNG_MAP[b] = p                          # key with extension
    PNG_MAP[os.path.splitext(b)[0]] = p     # key without extension
    k = _norm_img_key(b)                    # normalized key
    if k:
        PNG_MAP[k] = p

ALL_LABELS = ['cardiomegaly', 'pleural_effusion', 'pneumonia_opacity',
              'pneumothorax', 'edema', 'atelectasis', 'no_acute_finding']

def image_paths_for(row):
    out = []
    for i in str(row['image_ids']).split('|'):
        i = i.strip()
        if not i:
            continue
        if i in PNG_MAP:
            out.append(PNG_MAP[i])
            continue
        k = _norm_img_key(i)
        if k and k in PNG_MAP:
            out.append(PNG_MAP[k])
    return out

df['n_avail_imgs'] = df.apply(lambda r: len(image_paths_for(r)), axis=1)
n_noimg = int((df['n_avail_imgs'] == 0).sum())
print('reports with no image file found:', n_noimg)
if n_noimg == len(df):
    raise AssertionError(
        'NO report matched any PNG file — image-id/filename mismatch.\n'
        f'  image id sample : {str(df["image_ids"].iloc[0]).split("|")[:2]}\n'
        f'  png name sample : {[os.path.basename(p) for p in PNG_FILES[:2]]}\n'
        'Paste both sample lines back so the join can be fixed.')

df_test  = df[df['split'] == 'test'].reset_index(drop=True)
df_train = df[df['split'] == 'train'].reset_index(drop=True)
TEST_FILES = df_test['report_file'].tolist()          # fixed evaluation order
print(f'train reports: {len(df_train)} | test reports: {len(df_test)}')


In [ ]:
# =====================================================================
# STAGE 3.2 — Secondary metrics: BLEU-4, ROUGE-L, BERTScore
# (text overlap only — known to be weak for clinical correctness; they
#  are reported because the literature reports them, then contrasted
#  with the clinical-label metrics of 3.3)
# =====================================================================
from nltk.translate.bleu_score import corpus_bleu
from rouge_score import rouge_scorer
import bert_score

_rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def _tok(s):
    return re.findall(r'[a-z0-9]+', s.lower())

def bleu4(hyps, refs):
    return corpus_bleu([[_tok(r)] for r in refs], [_tok(h) for h in hyps])

def rougeL_f(hyps, refs):
    return float(np.mean([_rouge.score(r, h)['rougeL'].fmeasure
                          for r, h in zip(refs, hyps)]))

def bertscore_f1(hyps, refs):
    P, R, F1 = bert_score.score(list(hyps), list(refs), lang='en',
                                device='cuda' if torch.cuda.is_available() else 'cpu',
                                batch_size=32, verbose=False)
    return float(F1.mean())

print('sanity:', round(bleu4(['no acute cardiopulmonary abnormality'],
                            ['no acute cardiopulmonary abnormality']), 3), '(expect 1.0)')


In [ ]:
# =====================================================================
# STAGE 3.3 — PRIMARY metric: clinical-label precision/recall/F1
# Same extractor as Stage 2.6 applied to generated text; scores are
# computed against extractor labels of the REFERENCE reports. Extractor
# errors were measured on the manual validation sample (Stage 2.8) —
# the error rate must be quoted next to these numbers.
# NOTE: labels absent from BOTH sides contribute F1=0 to the macro
# (CheXpert-style), so macro-F1 is comparable ACROSS systems but is
# not an 'accuracy': identical texts score 1/7 unless all 7 labels are
# present. Present labels must still score exactly 1.0 — that is the
# self-check below.
# =====================================================================
from sklearn.metrics import precision_recall_fscore_support

def to_bin(status_dict):
    return {f: int(status_dict.get(f) == 'positive') for f in ALL_LABELS}

def binarize(texts):
    return pd.DataFrame([to_bin(label_report_full(t)[0]) for t in texts])[ALL_LABELS]

def clinical_eval(hyps, refs, hyp_bin=None, ref_bin=None):
    '''Per-label P/R/F1 + macro-F1. Bins can be precomputed and passed
    in (the bootstrap reuses them instead of re-running the extractor).'''
    if hyp_bin is None:
        hyp_bin = binarize(hyps)
    if ref_bin is None:
        ref_bin = binarize(refs)
    rows = {}
    for f in ALL_LABELS:
        p, r_, f1, _ = precision_recall_fscore_support(ref_bin[f], hyp_bin[f],
                                                       average='binary', zero_division=0)
        rows[f] = {'precision': p, 'recall': r_, 'f1': f1,
                   'ref_prevalence': float(ref_bin[f].mean())}
    macro = float(np.mean([v['f1'] for v in rows.values()]))
    return pd.DataFrame(rows).T, macro

# sanity: identical texts must agree perfectly on every PRESENT label
_t = 'small left pleural effusion. no pneumothorax.'
_per, _macro = clinical_eval([_t], [_t])
assert _per.loc['pleural_effusion', 'f1'] == 1.0, 'clinical metric self-check failed'
print('clinical metric self-check OK (present-label F1 = 1.0 on identical texts;')
print(f'macro over all 7 labels = {_macro:.3f} because absent labels count as 0)')

In [ ]:
# =====================================================================
# STAGE 3.4 — Bootstrap confidence intervals (as in the SQLoRA project)
# Design note: ROUGE-L here is a MEAN of per-report scores and the
# clinical macro-F1 comes from per-report label vectors — so their
# bootstraps resample PRECOMPUTED per-report quantities (seconds, not
# hours). Corpus BLEU-4 aggregates n-gram counts corpus-wide and cannot
# be decomposed per report, so it is recomputed each replicate.
# BERTScore is excluded from bootstrap (compute cost) — a stated
# limitation.
# =====================================================================
def _boot_indices(n, n_boot=1000, seed=SEED):
    rng = np.random.RandomState(seed)
    return [rng.randint(0, n, size=n) for _ in range(n_boot)]

def ci_of_means(per_report_values, n_boot=1000, seed=SEED):
    vals = np.asarray(per_report_values, dtype=float)
    means = [vals[idx].mean() for idx in _boot_indices(len(vals), n_boot, seed)]
    lo, hi = np.percentile(means, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)

def ci_corpus_bleu(hyps, refs, n_boot=1000, seed=SEED):
    idxs = _boot_indices(len(hyps), n_boot, seed)
    vals = [bleu4([hyps[i] for i in idx], [refs[i] for i in idx]) for idx in idxs]
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)

def _binary_f1(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    if tp == 0:
        return 0.0
    p = tp / (tp + fp)
    r = tp / (tp + fn)
    return 2 * p * r / (p + r)

def ci_clinical_macro(ref_bin, hyp_bin, n_boot=1000, seed=SEED):
    ref_m = ref_bin[ALL_LABELS].to_numpy()
    hyp_m = hyp_bin[ALL_LABELS].to_numpy()
    vals = []
    for idx in _boot_indices(len(ref_m), n_boot, seed):
        f1s = [_binary_f1(ref_m[idx, c], hyp_m[idx, c]) for c in range(ref_m.shape[1])]
        vals.append(float(np.mean(f1s)))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)

print('bootstrap helpers ready (precomputed-pair design)')

In [ ]:
# =====================================================================
# STAGE 3.5 — Generation registry + one 'evaluate this system' function
# Every system saves its test generations to Drive/notebook-output as
# JSON, so stages can be re-run independently without recomputing
# everything. _already_evaluated() lets a full Run-All skip systems
# already measured in an earlier session.
# =====================================================================
GEN_DIR = os.path.join(PERSIST, 'generations')
os.makedirs(GEN_DIR, exist_ok=True)
REF_TEXTS = df_test.set_index('report_file')['target_text']

def save_gen(name, records):
    with open(os.path.join(GEN_DIR, name + '.json'), 'w') as f:
        json.dump(records, f)
    print(f'saved generations: {name} ({len(records)} reports)')

def load_gen(name):
    p = os.path.join(GEN_DIR, name + '.json')
    if not os.path.exists(p):
        return None
    with open(p) as f:
        return json.load(f)

RESULTS_PATH = os.path.join(PERSIST, 'results_table.csv')

def _read_results_table():
    '''Read the results table, tolerating "not created yet" and an
    interrupted (empty) file — both yield a fresh empty table.'''
    if not os.path.exists(RESULTS_PATH):
        return pd.DataFrame()
    try:
        return pd.read_csv(RESULTS_PATH)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def _already_evaluated(name):
    '''True if the results table already carries a row for `name`.'''
    t = _read_results_table()
    return 'system' in t.columns and name in t['system'].values

def evaluate_system(name, n_boot=1000, with_ci=True):
    '''Compute all metrics for one saved system and append to the table.'''
    records = load_gen(name)
    assert records is not None, f'no generations saved for {name}'
    hyp_map = {r['report_file']: r for r in records}
    hyps = [hyp_map[f]['hyp'] for f in TEST_FILES]
    refs = [REF_TEXTS[f] for f in TEST_FILES]
    lat  = float(np.mean([hyp_map[f].get('latency_s', np.nan) for f in TEST_FILES]))
    hyp_bin, ref_bin = binarize(hyps), binarize(refs)
    per_label, macro = clinical_eval(hyps, refs, hyp_bin=hyp_bin, ref_bin=ref_bin)
    row = {'system': name,
           'BLEU-4': bleu4(hyps, refs),
           'ROUGE-L': rougeL_f(hyps, refs),
           'BERTScore-F1': bertscore_f1(hyps, refs),
           'clinical_macro-F1': macro,
           'mean_latency_s': lat}
    per_label.to_csv(os.path.join(PERSIST, f'clinical_per_label_{name}.csv'))
    if with_ci:
        per_pair_rouge = [_rouge.score(r, h)['rougeL'].fmeasure
                          for r, h in zip(refs, hyps)]
        row['BLEU-4_CI'] = ci_corpus_bleu(hyps, refs, n_boot)
        row['ROUGE-L_CI'] = ci_of_means(per_pair_rouge, n_boot)
        row['macroF1_CI'] = ci_clinical_macro(ref_bin, hyp_bin, n_boot)
    table = _read_results_table()
    if 'system' in table.columns:      # first-ever call: empty table, no columns
        table = table[table['system'] != name]
    table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    table.to_csv(RESULTS_PATH, index=False)
    print(name, '->', {k: (round(v, 4) if isinstance(v, float) else v)
                       for k, v in row.items() if k != 'system'})
    return row

print('registry ready — generations dir:', GEN_DIR)

In [ ]:
# =====================================================================
# STAGE 3.6 — Baseline B1: constant template report
# Template = the most frequent target text among NORMAL train reports
# (extractor labels all-zero on findings, positive on no_acute_finding).
# If data says otherwise, the template is what the data says — printed.
# B1_TEMPLATE is always (re)computed — later stages use it as the
# no-image fallback — but the generations are saved only once.
# =====================================================================
lab_idx = df_labels.set_index('report_file')
df_train = df_train.assign(
    is_normal=df_train['report_file']
        .map(lab_idx['bin_no_acute_finding']).fillna(0).astype(int) == 1)
for f in ALL_LABELS[:-1]:
    df_train['is_normal'] &= df_train['report_file'] \
        .map(lab_idx['bin_' + f]).fillna(0).astype(int) == 0

normal_texts = df_train.loc[df_train['is_normal'], 'target_text']
print(f'normal-labelled train reports: {len(normal_texts)} / {len(df_train)}')
if len(normal_texts):
    B1_TEMPLATE = normal_texts.mode().iloc[0]
    freq = (normal_texts == B1_TEMPLATE).mean()
    print(f'most common normal text covers {freq:.0%} of normal train reports')
else:
    B1_TEMPLATE = 'No acute cardiopulmonary abnormality.'
print('B1 template:', repr(B1_TEMPLATE))

if load_gen('B1_template') is None:
    save_gen('B1_template', [{'report_file': f, 'hyp': B1_TEMPLATE, 'latency_s': 0.0}
                              for f in TEST_FILES])
else:
    print('[skip] B1_template generations already saved')

In [ ]:
# =====================================================================
# STAGE 3.7 — Baseline B2 (1/2): CLIP image embeddings, cached to Drive
# CLIP ViT-B/32: small, ungated, trained contrastively on web image-text
# pairs — a reasonable off-the-shelf visual similarity for X-rays.
# One embedding per image; a study's embedding = mean of its images.
# =====================================================================
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(CLIP_ID).cuda().eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_ID)

EMB_CACHE = os.path.join(PERSIST, 'clip_image_embeddings.npz')

@torch.no_grad()
def embed_all_images(batch=64):
    if os.path.exists(EMB_CACHE):
        z = np.load(EMB_CACHE)
        return {k: z[k] for k in z.files}
    feats = {}
    paths = sorted(set(PNG_MAP.values()))   # PNG_MAP holds 2 keys per file
                                             # (with and without '.png') — dedupe
    for i in tqdm(range(0, len(paths), batch), desc='CLIP embeddings'):
        chunk = paths[i:i + batch]
        try:
            ims = [Image.open(p).convert('RGB') for p in chunk]
        except OSError:
            continue
        inp = clip_proc(images=ims, return_tensors='pt').to('cuda')
        f = clip.get_image_features(**inp)
        f = f / f.norm(dim=-1, keepdim=True)
        for p, v in zip(chunk, f.float().cpu().numpy()):
            feats[os.path.basename(p)] = v
    np.savez(EMB_CACHE, **feats)
    return feats

IMG_EMB = embed_all_images()
print('embedded images:', len(IMG_EMB))
del clip, clip_proc
torch.cuda.empty_cache()


In [ ]:
# =====================================================================
# STAGE 3.8 — Baseline B2 (2/2): nearest-training-study retrieval,
# then evaluate B1 and B2. Test studies with no available image fall
# back to the B1 template (counted and printed).
# RUN-ALL-SAFE: if B2 generations already exist (earlier session),
# the retrieval pass is skipped; evaluations run only for systems not
# yet in the results table.
# =====================================================================
def study_emb(row):
    vecs = [IMG_EMB[os.path.basename(p)] for p in image_paths_for(row)
            if os.path.basename(p) in IMG_EMB]
    if not vecs:
        return None
    v = np.mean(vecs, axis=0)
    return v / np.linalg.norm(v)

if load_gen('B2_retrieval') is None:
    train_mat, train_files = [], []
    for _, r in tqdm(df_train.iterrows(), total=len(df_train), desc='train embeddings'):
        v = study_emb(r)
        if v is not None:
            train_mat.append(v)
            train_files.append(r['report_file'])
    assert len(train_mat) > 0, ('no train study got an embedding — the image-id/PNG '
                                'join is broken (see the Stage 3.1 diagnostics)')
    train_mat = np.stack(train_mat)
    print('train studies embedded:', len(train_files))

    records, n_fallback = [], 0
    target_map = df_train.set_index('report_file')['target_text']
    for _, r in tqdm(df_test.iterrows(), total=len(df_test), desc='B2 retrieval'):
        v = study_emb(r)
        if v is None:
            records.append({'report_file': r['report_file'], 'hyp': B1_TEMPLATE,
                            'latency_s': 0.0})
            n_fallback += 1
            continue
        sims = train_mat @ v
        best = train_files[int(np.argmax(sims))]
        records.append({'report_file': r['report_file'], 'hyp': target_map[best],
                        'latency_s': 0.0})
    print(f'B2 test studies falling back to template (no image): {n_fallback}')
    save_gen('B2_retrieval', records)
else:
    print('[skip] B2_retrieval generations already saved')

for _name in ('B1_template', 'B2_retrieval'):
    if _already_evaluated(_name):
        print(f'[skip] {_name} already in the results table')
    else:
        evaluate_system(_name)

# STAGE 4 — VLM setup: 4-bit load, B3 zero-shot, and the 100-sample smoke test

**Model: `Qwen/Qwen2.5-VL-3B-Instruct`.** Ungated (no access-gate risk); ~4B parameters, so the 4-bit weights (~2.5 GB) + LoRA gradients + activations fit free 16 GB T4 GPUs (Colab T4; on Kaggle T4 ×2 the model is sharded across the two GPUs); native multi-image input lets a study's frontal and lateral views be passed together; the `transformers` integration is stable (>= 4.49). Fallback considered: `google/medgemma-4b-it` (radiology-oriented pretraining, but gated behind licence acceptance — access risk).

**Compute dtype note:** the T4 supports no bfloat16, so the 4-bit compute dtype falls back to float16 — chosen at runtime, not hard-coded.

**Modal (fine-tuning runs):** a single A10G 24 GB holds the whole 4-bit model + adapters without sharding, and Ampere-class GPUs add bf16 support (the runtime dtype check picks it automatically).

In [ ]:
# =====================================================================
# STAGE 4.1 — Load the VLM in 4-bit (NF4, double quant) + generation fn
# MAX_PIXELS caps each image at 448x448 -> ~256 visual tokens: the
# single biggest lever for training memory on a 16 GB GPU.
# =====================================================================
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_PIXELS = 448 * 448
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('compute dtype:', COMPUTE_DTYPE)

def load_base_4bit():
    bnb = BitsAndBytesConfig(load_in_4bit=True,
                             bnb_4bit_quant_type='nf4',
                             bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=COMPUTE_DTYPE)
    m = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map='auto')
    m.eval()
    return m

processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=MAX_PIXELS)

def build_messages(image_paths, instruction):
    content = [{'type': 'image'} for _ in image_paths]
    content.append({'type': 'text', 'text': instruction})
    return [{'role': 'user', 'content': content}]

@torch.no_grad()
def generate_report(model, image_paths, instruction, max_new_tokens=120):
    prompt = processor.apply_chat_template(
        build_messages(image_paths, instruction), tokenize=False,
        add_generation_prompt=True)
    ims = [Image.open(p).convert('RGB') for p in image_paths]
    inputs = processor(text=[prompt], images=ims, return_tensors='pt').to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         do_sample=False)
    dt = time.time() - t0
    txt = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:],
                                skip_special_tokens=True)[0].strip()
    return txt, dt

model = load_base_4bit()
print('peak GPU mem after load: %.1f GB' % (torch.cuda.max_memory_allocated() / 1e9))
# probe row: must HAVE images, else the processor call is meaningless
_probe = df_test[df_test['n_avail_imgs'] > 0].iloc[0]
_txt, _dt = generate_report(model, image_paths_for(_probe),
                            'Describe this image in one sentence.')
print(f'smoke generation ({_dt:.1f}s):', _txt[:200])

In [ ]:
# =====================================================================
# STAGE 4.2 — Baseline B3: zero-shot VLM with a FIXED documented prompt
# Test studies with no available image (counted, printed) fall back to
# the B1 template so every test report gets a generation.
# RUN-ALL-SAFE: the ~1.5 h generation pass runs only if B3 generations
# do not already exist; the evaluation runs only if B3 is not yet in
# the results table.
# =====================================================================
PROMPT_B3 = ('You are a radiologist reading a chest X-ray examination. '
             'Write the Findings section of the report for this study. '
             'Describe only what you can see in the image(s). '
             'If there is no abnormality, write exactly: '
             'No acute cardiopulmonary abnormality.')
print('B3 prompt (fixed):', PROMPT_B3)

if load_gen('B3_zeroshot') is None:
    N_B3_LIMIT = None        # e.g. 100 for a quick pass; None = full test set
    rows = df_test if N_B3_LIMIT is None else df_test.head(N_B3_LIMIT)
    records, n_fb = [], 0
    for _, r in tqdm(rows.iterrows(), total=len(rows), desc='B3 zero-shot'):
        paths = image_paths_for(r)
        if not paths:
            records.append({'report_file': r['report_file'], 'hyp': B1_TEMPLATE,
                            'latency_s': 0.0})
            n_fb += 1
            continue
        txt, dt = generate_report(model, paths, PROMPT_B3)
        records.append({'report_file': r['report_file'], 'hyp': txt, 'latency_s': dt})
    print(f'B3 studies with no image (template fallback): {n_fb}')
    save_gen('B3_zeroshot', records)
else:
    print('[skip] B3_zeroshot generations already saved')

if _already_evaluated('B3_zeroshot'):
    print('[skip] B3_zeroshot already in the results table')
else:
    evaluate_system('B3_zeroshot')

del model
torch.cuda.empty_cache()

In [ ]:
# =====================================================================
# STAGE 4.3 — Training-data builder + LoRA attach + 100-SAMPLE SMOKE TEST
# Run BEFORE any full training: proves the memory fit, the loss goes
# down, and gives a per-step timing to extrapolate the full run.
# Training uses only reports with at least one available image (counted).
# =====================================================================
from peft import LoraConfig, get_peft_model

PROMPT_GEN = ('You are a radiologist. Write the Findings section of the '
              'report for this chest X-ray study.')
LORA_TARGETS = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                'gate_proj', 'up_proj', 'down_proj']   # LLM blocks; the vision
                                                        # tower (qkv/proj) is untouched

def build_train_example(model, row):
    '''Returns model inputs with the prompt tokens masked out of labels.'''
    paths = image_paths_for(row)
    ims = [Image.open(p).convert('RGB') for p in paths]
    prompt = processor.apply_chat_template(
        build_messages(paths, PROMPT_GEN), tokenize=False,
        add_generation_prompt=True)
    full = prompt + row['target_text'] + processor.tokenizer.eos_token
    inputs = processor(text=[full], images=ims, return_tensors='pt').to(model.device)
    prompt_len = processor(text=[prompt], images=ims,
                           return_tensors='pt')['input_ids'].shape[1]
    labels = inputs['input_ids'].clone()
    labels[:, :prompt_len] = -100
    inputs['labels'] = labels
    return inputs

def attach_lora(model, r):
    cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias='none',
                     task_type='CAUSAL_LM', target_modules=LORA_TARGETS)
    peft_model = get_peft_model(model, cfg)
    n_train = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in peft_model.parameters())
    print(f'LoRA r={r}: trainable {n_train/1e6:.2f}M '
          f'({100*n_train/n_total:.2f}% of {n_total/1e9:.2f}B)')
    return peft_model, n_train

# ---------------- smoke test: 100 train reports, one pass --------------
train_usable = df_train[df_train['n_avail_imgs'] > 0]
print(f'train reports usable for training (image available): '
      f'{len(train_usable)} / {len(df_train)}')
smoke = train_usable.sample(n=100, random_state=SEED).reset_index(drop=True)
model = load_base_4bit()
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()
model, _n = attach_lora(model, r=32)
model.train()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=2e-4, weight_decay=0.01)
torch.cuda.reset_peak_memory_stats()
losses, t0 = [], time.time()
for i, row in smoke.iterrows():
    inputs = build_train_example(model, row)
    loss = model(**inputs).loss
    loss.backward()
    opt.step(); opt.zero_grad()
    losses.append(float(loss.item()))
    if i in (0, 4, 9, 49, 99):
        print(f'step {i+1:3d}  loss {losses[-1]:.3f}  '
              f'peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB')
per_step = (time.time() - t0) / len(smoke)
print(f'\nsmoke: first10 mean {np.mean(losses[:10]):.3f} -> '
      f'last10 mean {np.mean(losses[-10:]):.3f}')
assert np.isfinite(losses).all(), 'non-finite loss — debug before full training'
print(f'per-step {per_step:.2f}s -> 2 epochs on {len(train_usable)} reports ~= '
      f'{per_step*2*len(train_usable)/3600:.1f} h')
SMOKE_PER_STEP = per_step      # measured speed — Stage 5.2 uses it to
                               # auto-cap epochs for an unattended run

del model, opt
torch.cuda.empty_cache()

# STAGE 5 — QLoRA fine-tuning (main contribution) and the LoRA-rank ablation

**Configuration and why:** LoRA r=32 (α=64, dropout 0.05) on the 7 linear projections of the *language* blocks — the vision tower stays frozen because IU X-Ray (≈3k studies) is far too small to adapt a vision encoder without overfitting. QLoRA (NF4 + double quantisation; compute dtype fp16 on T4, bf16 on Ampere-class GPUs — chosen at runtime) because only the adapter gradients are kept in memory. AdamW lr 2e-4, cosine schedule, effective batch 8 (batch 1 × accumulation 8 — variable image token counts make batch 1 the robust choice), 2 epochs, gradient clipping 1.0. Greedy decoding at inference; the train and inference prompts are the same string by construction.

**Collapse incident and mitigation (documented honestly):** the first complete run of this pipeline (2026-10-06/07, identical configuration except unbalanced sampling) collapsed: the fine-tuned model emitted one generic normal report for every image — clinical macro-F1 **0.00** (it never asserts a finding), while also never hallucinating one (0 false positives across all 766 test reports). The training code passes the images correctly (verified in the executed notebook), so the cause is the majority-class shortcut of a normal-dominated dataset with a frozen vision tower. The single mitigation, deliberately the only change vs that run: **class-balanced training** — abnormal reports (target text with at least one positive extractor label) are oversampled up to 3x per epoch so the model sees roughly as many abnormal as normal targets. Everything else (LoRA config, lr, schedule, prompt, decoding) is untouched, so the before/after comparison isolates the effect of the sampling fix.

**v4 addendum (second incident, documented like the first):** the first balanced run (2026-10-07 v4) hit a **non-finite loss ~1 300 iterations in** — a bf16 numerical instability on a single batch. With no guard, the NaN propagated into the LoRA weights and every remaining iteration was lost (the train log's loss curve is NaN from that point; the run was cancelled). `run_training` now applies the standard robust-training guard: any batch with a non-finite loss and any optimizer step with a non-finite gradient norm are **skipped, counted and printed** (`n_skipped_batches` in the train log) — nothing else changed.

**Ablation (single, by design):** LoRA rank 32 → 8 with everything else fixed. Rank is the cleanest one-knob question ("does adapter capacity matter at this data scale?") and costs exactly one extra training run. Frontal-vs-frontal+lateral and prompt-format ablations were deliberately cut to keep the compute budget honest.

In [ ]:
# =====================================================================
# STAGE 5.1 — Training loop (fresh 4-bit base each run, adapter to Drive)
# Trains only on reports with an available image (same filter as the
# smoke test), so training never crashes on a missing PNG.
# Class-balanced pool (see the Stage 5 header): abnormal reports are
# oversampled up to 3x per epoch — the anti-collapse mitigation.
# The adapter is checkpointed to PERSIST (Drive / notebook output /
# Modal volume — background commits make each write durable) every
# 750 iterations: if the session dies mid-run, a usable adapter
# checkpoint still exists.
# =====================================================================
from transformers import get_cosine_schedule_with_warmup

def run_training(lora_r, tag, epochs=2, accum=8, lr=2e-4, seed=SEED,
                 log_every=100, limit=None, checkpoint_every=750,
                 balance=True):
    torch.manual_seed(seed)
    pool = df_train[df_train['n_avail_imgs'] > 0].reset_index(drop=True)
    n_abn = n_nor = None
    factor = 1
    if balance:
        _abn = pool['target_text'].map(
            lambda t: any(s == 'positive' for s in label_text(t)[0].values()))
        n_abn, n_nor = int(_abn.sum()), int((~_abn).sum())
        factor = min(int(np.ceil(n_nor / max(n_abn, 1))), 3)
        pool = pd.concat([pool[~_abn]] + [pool[_abn]] * factor,
                         ignore_index=True)
        print(f'[{tag}] balance: {n_abn} abnormal x{factor} + {n_nor} normal '
              f'= {len(pool)} reports/epoch')
    data = pool if limit is None else pool.head(limit)
    adapter_dir = os.path.join(PERSIST, 'adapters', tag)
    os.makedirs(adapter_dir, exist_ok=True)
    model = load_base_4bit()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={'use_reentrant': False})
    model.enable_input_require_grads()
    model, n_train_params = attach_lora(model, lora_r)
    model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    total_steps = math.ceil(len(data) * epochs / accum)
    sched = get_cosine_schedule_with_warmup(opt, 30, total_steps)
    torch.cuda.reset_peak_memory_stats()
    losses, t0 = [], time.time()
    n_bad = 0
    for ep in range(epochs):
        order = np.random.RandomState(seed + ep).permutation(len(data))
        for j, idx in enumerate(order):
            inputs = build_train_example(model, data.iloc[idx])
            loss = model(**inputs).loss / accum
            if not torch.isfinite(loss):
                n_bad += 1
                opt.zero_grad()
                print(f'[{tag}] non-finite loss at it {j} — batch skipped '
                      f'({n_bad} total)', flush=True)
                continue
            loss.backward()
            losses.append(float(loss.item()) * accum)
            if (j + 1) % accum == 0:
                _norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
                if torch.isfinite(_norm):
                    opt.step(); sched.step()
                else:
                    n_bad += 1
                    print(f'[{tag}] non-finite grad norm at it {j} — step '
                          f'skipped ({n_bad} total)', flush=True)
                opt.zero_grad()
            if j % log_every == 0:
                print(f'[{tag}] ep{ep} it{j:5d}/{len(data)} '
                      f'loss {np.mean(losses[-50:]):.3f} '
                      f'peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB '
                      f'elapsed {(time.time()-t0)/3600:.2f} h', flush=True)
            if (j + 1) % checkpoint_every == 0:
                model.save_pretrained(adapter_dir)
                print(f'[{tag}] intermediate adapter checkpoint saved '
                      f'(it {j+1}, {(time.time()-t0)/3600:.2f} h)', flush=True)
    seconds = time.time() - t0
    model.save_pretrained(adapter_dir)
    log = {'tag': tag, 'lora_r': lora_r, 'epochs': epochs,
           'balanced': bool(balance), 'n_abnormal': n_abn, 'n_normal': n_nor,
           'oversample_factor': factor, 'n_skipped_batches': n_bad,
           'n_train_reports': len(data), 'trainable_params': int(n_train_params),
           'seconds': seconds, 'hours': seconds / 3600,
           'peak_gpu_gb': torch.cuda.max_memory_allocated() / 1e9,
           'final_loss_50': float(np.mean(losses[-50:])),
           'first_loss_50': float(np.mean(losses[:50])),
           'loss_curve_step100': losses[::100]}
    with open(os.path.join(PERSIST, f'train_log_{tag}.json'), 'w') as f:
        json.dump(log, f)
    print(f"[{tag}] done in {log['hours']:.2f} h, adapter -> {adapter_dir}")
    del model, opt, sched
    torch.cuda.empty_cache()
    return adapter_dir, log

print('run_training ready')

In [ ]:
# =====================================================================
# STAGE 5.2 — MAIN RUN: QLoRA r=32 on all usable train reports
# RUN-ALL-SAFE: if train_log_m_r32.json exists (earlier session), the
# run is skipped and the log is loaded so downstream stages see it.
# Epochs are AUTO-CAPPED from the measured smoke speed so an unattended
# overnight run finishes: if both runs at 2 epochs project to > 6 h of
# training in total, BOTH runs use 1 epoch (same value for main and
# ablation — the comparison stays fair; the decision is printed and
# also lands in the train logs). To force a value, set EPOCHS_M manually.
# The try/except keeps an unattended Run-all moving if this run fails:
# the failure is PRINTED (never hidden) and later cells say clearly
# what is missing.
# =====================================================================
import traceback

if os.path.exists(os.path.join(PERSIST, 'train_log_m_r32.json')):
    with open(os.path.join(PERSIST, 'train_log_m_r32.json')) as f:
        LOG_M = json.load(f)
    ADAPTER_M = os.path.join(PERSIST, 'adapters', 'm_r32')
    EPOCHS_M = int(LOG_M.get('epochs', 1))
    print('[skip] m_r32 already trained:', {k: LOG_M[k] for k in
                                            ('tag', 'lora_r', 'epochs', 'hours')})
else:
    EPOCHS_M = 2
    _abn = train_usable['target_text'].map(
        lambda t: any(s == 'positive' for s in label_text(t)[0].values()))
    _n_abn, _n_nor = int(_abn.sum()), int((~_abn).sum())
    _bal = _n_nor + min(int(np.ceil(_n_nor / max(_n_abn, 1))), 3) * _n_abn
    _projected_h = SMOKE_PER_STEP * EPOCHS_M * _bal * 2 / 3600
    print(f'train pool: {_n_abn} abnormal / {_n_nor} normal -> balanced '
          f'epoch = {_bal} reports (oversample up to 3x, cap 3)')
    print(f'projected TRAINING time, 2 runs x {EPOCHS_M} epoch(s) on the '
          f'balanced pool: {_projected_h:.1f} h')
    if _projected_h > 6:
        EPOCHS_M = 1
        print(f'AUTO-DECISION: projection > 6 h -> EPOCHS_M = 1 so the full '
              f'pipeline finishes overnight (both runs use the same value)')
    try:
        ADAPTER_M, LOG_M = run_training(lora_r=32, tag='m_r32', epochs=EPOCHS_M)
    except Exception:
        print('!! m_r32 training FAILED — continuing so the night is not lost:')
        traceback.print_exc()

In [ ]:
# =====================================================================
# STAGE 5.3 — ABLATION RUN: identical except LoRA r=8
# RUN_ABLATION is THE one switch:
#   False -> this cell only prints a [skip] line (single-model session)
#   True  -> trains m_r8 in the SAME session right after m_r32 (the
#            one-session plan on a pay-per-second rented GPU), or skips
#            if train_log_m_r8.json already exists from an earlier run.
# epochs always inherit EPOCHS_M from the m_r32 run/log, so the main
# vs ablation comparison stays matched by construction.
# (same guard: a failure here must not stop the generation stage)
# =====================================================================
RUN_ABLATION = True

if not RUN_ABLATION:
    print('[skip] ablation m_r8 disabled this session (RUN_ABLATION = False)')
elif os.path.exists(os.path.join(PERSIST, 'train_log_m_r8.json')):
    with open(os.path.join(PERSIST, 'train_log_m_r8.json')) as f:
        LOG_A = json.load(f)
    ADAPTER_A = os.path.join(PERSIST, 'adapters', 'm_r8')
    print('[skip] m_r8 already trained:', {k: LOG_A[k] for k in
                                           ('tag', 'lora_r', 'epochs', 'hours')})
else:
    try:
        ADAPTER_A, LOG_A = run_training(lora_r=8, tag='m_r8', epochs=EPOCHS_M)
    except Exception:
        print('!! m_r8 training FAILED — continuing so the night is not lost:')
        traceback.print_exc()


In [ ]:
# =====================================================================
# STAGE 5.4 — Generate test reports with each fine-tuned adapter
# (same prompt string as training, greedy decoding, batch 1)
# Studies with no available image fall back to the B1 template
# (same rule as B3 — counted and printed, identical across systems).
# RESILIENCE (learned from a real session death on 2026-10-03):
# generations are saved to Drive/notebook-output INCREMENTALLY (every
# 100 reports) and a re-run RESUMES from the saved partial file, so a
# session death mid-generation costs only the unsaved tail.
# RUN-ALL-SAFE: an adapter whose generations are complete AND already
# evaluated is skipped entirely. Each adapter's generation is guarded
# so one failure cannot block the other or Stages 6–7.
# =====================================================================
import gc
import traceback
from peft import PeftModel

def generate_with_adapter(tag, save_every=100):
    out_path = os.path.join(GEN_DIR, tag + '.json')
    records, done = [], set()
    if os.path.exists(out_path):                     # resume a partial pass
        with open(out_path) as f:
            records = json.load(f)
        done = {r['report_file'] for r in records}
        print(f'{tag}: resuming — {len(done)} reports already generated')
    model = load_base_4bit()
    model = PeftModel.from_pretrained(
        model, os.path.join(PERSIST, 'adapters', tag))
    model.eval()
    n_fb = 0
    todo = df_test[~df_test['report_file'].isin(done)]
    for _, r in tqdm(todo.iterrows(), total=len(todo), desc=f'generate {tag}'):
        paths = image_paths_for(r)
        if not paths:
            records.append({'report_file': r['report_file'], 'hyp': B1_TEMPLATE,
                            'latency_s': 0.0})
            n_fb += 1
        else:
            txt, dt = generate_report(model, paths, PROMPT_GEN)
            records.append({'report_file': r['report_file'], 'hyp': txt,
                            'latency_s': dt})
        if len(records) % save_every == 0:            # incremental save
            with open(out_path, 'w') as f:
                json.dump(records, f)
    save_gen(tag, records)                           # final save + confirm
    print(f'{tag}: template fallbacks in this pass: {n_fb}')
    evaluate_system(tag)
    del model
    torch.cuda.empty_cache()

for _tag in ('m_r32', 'm_r8'):
    try:
        _done = load_gen(_tag)
        if _done is not None and len(_done) >= len(TEST_FILES) and _already_evaluated(_tag):
            print(f'[skip] {_tag}: generations and metrics already on record')
            continue
        generate_with_adapter(_tag)
    except Exception:
        print(f'!! generation with {_tag} FAILED — continuing:')
        traceback.print_exc()
        gc.collect()
        torch.cuda.empty_cache()

# STAGE 6 — Full evaluation: main results table with bootstrap CIs

Every number below is computed from the generations saved in `PERSIST/generations/`. Systems not yet run are skipped with a message — re-run this cell after Stages 3–5 to complete the table.


In [ ]:
# =====================================================================
# STAGE 6.1 — Main results table (all systems, all metrics, CIs)
# RUN-ALL-SAFE: systems whose generations are missing are skipped with
# a message; systems already evaluated are not recomputed.
# =====================================================================
SYSTEM_ORDER = ['B1_template', 'B2_retrieval', 'B3_zeroshot', 'm_r32', 'm_r8']
SYSTEM_NAMES = {'B1_template': 'B1 template', 'B2_retrieval': 'B2 retrieval',
                'B3_zeroshot': 'B3 zero-shot VLM', 'm_r32': 'M QLoRA r=32',
                'm_r8': 'Ablation QLoRA r=8'}

for name in SYSTEM_ORDER:
    if load_gen(name) is None:
        print(f'[skip] {name}: no generations yet')
        continue
    if not _already_evaluated(name):
        evaluate_system(name)

table = _read_results_table()
table['ord'] = table['system'].map(lambda s: SYSTEM_ORDER.index(s) if s in SYSTEM_ORDER else 99)
table = table.sort_values('ord').drop(columns='ord')
table['system'] = table['system'].map(SYSTEM_NAMES)
table.to_csv(os.path.join(PERSIST, 'main_results_table.csv'), index=False)
table

# STAGE 7 — Hallucination analysis, cost analysis, qualitative examples

Manual review is a *deliverable of the protocol*, not decoration: automatic metrics cannot diagnose *what kind* of unfaithfulness a model produces. 50 reports from the main model, stratified by the extractor's automatic FP/FN signals (the stratification speeds up review; the categories themselves are assigned by the author by reading reference + generation side by side against the image).


In [ ]:
# =====================================================================
# STAGE 7.1 — Build the 50-report manual hallucination-review sample
# Categories (fixed taxonomy, from the project brief):
#   FP-finding | FN-finding | wrong-laterality-location | contradiction |
#   fluent-ungrounded | none            (multiple allowed, separated by ';')
# GUARD: if the sample already exists on Drive WITH manual labels filled,
# it is kept as-is — re-running this cell must never destroy manual work.
# =====================================================================
_m32 = load_gen('m_r32')
assert _m32 is not None, ('no m_r32 generations on Drive — Stage 5.4 did not '
                          'complete (session died?). Re-run it, then Stage 7.')
recs = {r['report_file']: r for r in _m32}
ref_map = df_test.set_index('report_file')['target_text']

_hall_p = os.path.join(PERSIST, 'hallucination_sample.csv')
_keep = False
if os.path.exists(_hall_p):
    _prev = pd.read_csv(_hall_p)
    _mc = [c for c in _prev.columns if c.startswith('manual_')]
    _keep = any(_prev[c].fillna('').astype(str).str.strip().ne('').any()
                for c in _mc)

if _keep:
    print('hallucination_sample.csv already carries manual labels — kept (no re-sample)')
    sample = _prev
else:
    rows = []
    for f in TEST_FILES:
        hyp_bin = to_bin(label_report_full(recs[f]['hyp'])[0])
        ref_bin = to_bin(label_report_full(ref_map[f])[0])
        fp = [l for l in ALL_LABELS if hyp_bin[l] == 1 and ref_bin[l] == 0]
        fn = [l for l in ALL_LABELS if ref_bin[l] == 1 and hyp_bin[l] == 0]
        pool = 'FP' if fp else ('FN' if fn else 'clean')
        rows.append({'report_file': f, 'pool': pool, 'auto_fp': '|'.join(fp),
                     'auto_fn': '|'.join(fn), 'hyp': recs[f]['hyp'],
                     'ref': ref_map[f]})
    rev = pd.DataFrame(rows)
    print('pool sizes:', rev['pool'].value_counts().to_dict())

    rng = np.random.RandomState(SEED)
    parts = [rev[rev['pool'] == 'FP'].sample(min(25, (rev['pool'] == 'FP').sum()), random_state=rng),
             rev[rev['pool'] == 'FN'].sample(min(15, (rev['pool'] == 'FN').sum()), random_state=rng),
             rev[rev['pool'] == 'clean'].sample(min(10, (rev['pool'] == 'clean').sum()), random_state=rng)]
    sample = pd.concat(parts).sample(frac=1, random_state=rng)   # shuffle
    sample['manual_category'] = ''
    sample['manual_notes'] = ''
    sample.to_csv(_hall_p, index=False)
    print(f'saved -> PERSIST/hallucination_sample.csv ({len(sample)} reports)')
    print('author task: read each row (image + ref + hyp) and fill manual_category')
sample[['report_file', 'pool', 'auto_fp', 'auto_fn']].head(10)


In [ ]:
# =====================================================================
# STAGE 7.2 — Hallucination rates (runs only once manual_category is filled)
# =====================================================================
p = os.path.join(PERSIST, 'hallucination_sample.csv')
hs = pd.read_csv(p)
if hs['manual_category'].fillna('').str.strip().eq('').all():
    print('manual_category is empty — fill the CSV in Drive first, then re-run.')
else:
    done = hs[hs['manual_category'].fillna('').str.strip() != '']
    cats = done['manual_category'].str.split(';').explode().str.strip()
    cats = cats[cats != '']
    rates = (cats.value_counts() / len(done))
    print(f'manually reviewed: {len(done)} / {len(hs)}')
    print('rate per category (a report can carry several categories):')
    print((rates * 100).round(1).astype(str) + ' %')
    rates.to_frame('rate').to_csv(os.path.join(PERSIST, 'hallucination_rates.csv'))


In [ ]:
# =====================================================================
# STAGE 7.3 — Cost & efficiency table (explicitly required by the brief)
# =====================================================================
assert os.path.exists(RESULTS_PATH), (
    'results_table.csv not found — run Stage 6 (evaluate_system) first')
cost_rows = [
    {'system': 'B1 template',   'trainable_params': 0, 'train_hours': 0.0,
     'peak_gpu_gb': 0.0, 'mean_latency_s': 0.0},
    {'system': 'B2 retrieval',  'trainable_params': 0, 'train_hours': 0.0,
     'peak_gpu_gb': float('nan'), 'mean_latency_s': float('nan')},
]
table = pd.read_csv(RESULTS_PATH)
def _lat(name):
    if name in table['system'].values:
        return float(table.loc[table['system'] == name, 'mean_latency_s'].iloc[0])
    return float('nan')
for tag, disp in [('m_r32', 'M QLoRA r=32'), ('m_r8', 'Ablation r=8')]:
    lp = os.path.join(PERSIST, f'train_log_{tag}.json')
    if os.path.exists(lp):
        with open(lp) as f:
            lg = json.load(f)
        cost_rows.append({'system': disp,
                          'trainable_params': lg['trainable_params'],
                          'train_hours': round(lg['hours'], 2),
                          'peak_gpu_gb': round(lg['peak_gpu_gb'], 2),
                          'mean_latency_s': _lat(tag)})
cost = pd.DataFrame(cost_rows)
cost['trainable_params_M'] = (cost['trainable_params'] / 1e6).round(2)
cost['trainable_params_pct_of_4B'] = (cost['trainable_params'] / 4e9 * 100).round(2)
cost.to_csv(os.path.join(PERSIST, 'cost_table.csv'), index=False)
cost

In [ ]:
# =====================================================================
# STAGE 7.4 — Qualitative examples: one good, one average, one failure
# =====================================================================
_m32 = load_gen('m_r32')
assert _m32 is not None, ('no m_r32 generations on Drive — Stage 5.4 did not '
                          'complete. Re-run it first, then this cell.')
recs = {r['report_file']: r for r in _m32}
ref_map = df_test.set_index('report_file')['target_text']
qual = []
for f in TEST_FILES:
    r_ = _rouge.score(ref_map[f], recs[f]['hyp'])['rougeL'].fmeasure
    hyp_bin = to_bin(label_report_full(recs[f]['hyp'])[0])
    ref_bin = to_bin(label_report_full(ref_map[f])[0])
    fp = any(hyp_bin[l] > ref_bin[l] for l in ALL_LABELS)
    qual.append({'report_file': f, 'rougeL': r_, 'has_fp': fp})
qual = pd.DataFrame(qual)

def show_example(row, title):
    print('=' * 70)
    print(title, '| rougeL %.2f' % row['rougeL'], '|', row['report_file'])
    paths = image_paths_for(df_test.set_index('report_file').loc[row['report_file']])
    if paths:
        plt.figure(figsize=(4, 4))
        plt.imshow(Image.open(paths[0]), cmap='gray')
        plt.axis('off'); plt.show()
    print('REFERENCE:', ref_map[row['report_file']])
    print('GENERATED:', recs[row['report_file']]['hyp'])

show_example(qual.sort_values('rougeL', ascending=False).iloc[0], 'GOOD')
show_example(qual.sort_values('rougeL').iloc[len(qual) // 2], 'AVERAGE')
show_example(qual[qual['has_fp']].sort_values('rougeL').iloc[0]
             if qual['has_fp'].any() else qual.sort_values('rougeL').iloc[0],
             'FAILURE (extractor flags a false-positive finding)')

# STAGE 8 — Written report sections

All prose below is final: every results sentence was completed **from the tables produced above after the real v5 run** (master-prompt honesty rule: no number**from the tables produced above after the real runs** (master-prompt honesty rule: no number is written before it is computed). The Introduction and State of the Art are at the TOP of the notebook (sections 1–2, before Stage 1); this stage holds results, analysis, limits and references.

## 8.1 Methodology summary

- **Data:** IU X-Ray (Open-I), findings/impression sections, patient-level 70/10/20 split with hard leakage assertions (Stage 2); reports with no text or no linked image excluded (counted in Stage 2.3).
- **Systems:** B1 constant template (mode of normal train reports); B2 retrieval of the nearest train study via CLIP image embeddings; B3 zero-shot Qwen2.5-VL-3B with a fixed prompt; M = QLoRA r=32; ablation r=8 (Stage 5 rationale).
- **Primary evaluation:** clinical-label F1 via the Stage 2.6 extractor; **secondary:** BLEU-4 / ROUGE-L / BERTScore; bootstrap 95% CIs; manual hallucination taxonomy on 50 reports; cost table.
- **Extractor validation (completed 2026-10-08):** the author labelled all 60 sampled reports by hand (7 labels × P/N/U = 420 judgments, reading every report; the AI first-pass suggestions in `ai_hint_*` were confirmed or corrected — the final `manual_*` labels are the author's). Per-label agreement with the rule extractor: cardiomegaly 96.7%, pneumonia/opacity 96.7%, edema 95.0%, atelectasis 96.7%, pneumothorax 80.0%, pleural effusion 76.7%, *no acute finding* **48.3%**; **overall 354/420 = 84.3%**. The disagreements quantify exactly the two documented failure modes: 32/420 labels marked `positive` on text that negates or never asserts the finding (the “No X, Y or Z” list-negation gap), and 26/420 labels where an asserted state was left `unmentioned`/`uncertain` (dominated by *no acute finding*: “lungs are clear”-type phrasing). The same extractor is applied to references and to all five systems' outputs, so relative comparisons are far less exposed than absolute values.

## 8.2 Results — to complete from `main_results_table.csv`

Main table (766 test reports, bootstrap 95% CIs):

| system | BLEU-4 | ROUGE-L | BERTScore | clinical macro-F1 (95% CI) |
|---|---|---|---|---|
| B1 template | 0.0412 | 0.2383 | 0.8772 | 0.0384 (0.033–0.044) |
| B2 retrieval | 0.0771 | 0.2400 | 0.8752 | **0.1736** (0.139–0.208) |
| B3 zero-shot VLM | 0.0177 | 0.1740 | 0.8563 | 0.1518 (0.125–0.179) |
| M QLoRA r=32 | 0.0490 | 0.2458 | 0.8788 | 0.0838 (0.068–0.100) |
| Ablation QLoRA r=8 | 0.0598 | **0.2550** | **0.8807** | 0.0856 (0.070–0.100) |

1. **M clears B1 on the primary metric**: 0.0838 vs 0.0384, and the confidence intervals (0.068–0.100 vs 0.033–0.044) do not overlap — the adapted model extracts genuinely more clinical signal than a fixed normal template. It does **not** clear the stronger baselines: B2 retrieval (0.1736, CI 0.139–0.208) and B3 zero-shot (0.1518, CI 0.125–0.179) both rank above M with non-overlapping CIs against it.
2. **Per label** (clinical_per_label CSVs): M's macro-F1 comes almost entirely from three of seven classes — pneumonia/opacity F1 0.223, pneumothorax 0.080, and *no acute finding* 0.284 (recall 0.924, precision 0.168) — while cardiomegaly, pleural effusion, edema and atelectasis sit at **exactly 0.000**: the model never asserts them. B2, by contrast, spreads moderate F1 across **all seven** labels (0.089–0.269), because retrieval copies neighbour findings verbatim: breadth beats M's three-label precision.
3. **Overlap and clinical metrics disagree in ranking — the expected key finding, confirmed.** B3 is *last* on BLEU-4 (0.0177) yet *second* on clinical macro-F1 (0.1518); M r=8 is *first* on ROUGE-L (0.2550) and BERTScore (0.8807) yet only fourth on the primary metric. N-gram and embedding similarity reward fluent, normal-sounding prose that the clinical metric correctly discounts. Ranking these systems by BLEU alone would have inverted the conclusions.
4. **Ablation**: r=32 → 0.0838 vs r=8 → 0.0856, a 0.002 gap far inside the CI width (~±0.015 each side). LoRA rank is immaterial here; the outcome is not a capacity phenomenon — which strengthens the data/objective interpretation of the remaining gap to B2.

## 8.3 Critical analysis — to complete from Stage 7 outputs

**Hallucination pools (Stage 7, over 766 test reports):** 588 reports carry at least one false-positive assertion, 46 miss a referenced finding, 132 are fully clean; the 50-row manual-review sample is 25 FP + 15 FN + 10 clean. The author's completed manual review of those 50 reports (final labels, 2026-10-08): **28/50 (56%) extractor errors** — automatic flags that are artifacts of negation gaps or no-acute-finding phrasing asymmetries, not model hallucinations; **13/50 (26%) real hallucinations** — 11 silent misses of findings the reference asserts (worst case 91.xml: the model writes “There is no pneumothorax” against a reference that measures a 3.2 cm right pneumothorax) + 2 false normality claims (normal-sounding templates on abnormal reference films); **9/50 (18%) correct**. In other words, most automatic flags overstate hallucination behaviour, but the real ones cluster exactly where it matters clinically: silent misses of abnormal findings.

**Dominant failure mode — over-assertion of normality.** The collapse incident of the first run (v3: identical generic normal output, 0 false positives, 455 misses) was only partially fixed by class-balanced training: outputs diversified from 1 to 6 templates (86% still one sentence), and the error *flipped direction* — the no-acute-finding class is now asserted with recall 0.924 but precision 0.168, i.e. the model declares abnormal films "clear" (588/766). Clinically this is the more dangerous direction: a confident false "normal".

**Concrete example (from the FP sample):** report `1900.xml` — reference: *"…sternotomy … The aorta is calcified and tortuous. There are scattered calcified granulomas throughout both lungs…"* → hypothesis: *"The heart is normal in size. The mediastinum is unremarkable. The lungs are clear."* The automatic flags (FP `no_acute_finding`; FN `pleural_effusion|pneumothorax`) also expose a visible extractor negation gap on the phrase "No focal infiltrate, pleural effusion, or pneumothorax" — a second, honest reason the 60-report manual validation of the extractor matters before any absolute number is trusted. That validation is now complete: the author labelled all 420 cells (reading every report, confirming or correcting the AI first-pass suggestions), final agreement 354/420 = 84.3%, with 32 confirmed extractor positives on negated or unasserted text and 26 missed assertions — the two failure modes this section describes, now quantified.

**Cost table (real):** r=32 — 74.31 M trainable parameters (**1.86%** of the model), 2.80 h train, peak GPU 8.36 GB, 283.6 MiB adapter, 2.51 s/report greedy decoding; r=8 — 18.58 M (**0.46%**), 2.76 h, 7.39 GB, 71 MiB, 2.81 s/report. Robustness note: the non-finite-loss guard skipped 5 batches during the r=8 run (bf16 instability, printed in Stage 5 outputs; `n_skipped_batches` in the train log) — training survived what killed the v4 attempt. Practical reading: adaptation is cheap, adapters are tiny and shareable, inference is ~2.5 s on one GPU — the binding constraint on usefulness is clinical faithfulness, not compute.

## 8.4 Limits and perspectives

- **Data:** IU X-Ray is small (~3–4k studies) and single-institution; label prevalence is skewed towards normal studies, so rare-finding F1 is unstable. MIMIC-CXR would be the natural scale-up but requires credentialed access.
- **Metrics:** the clinical metrics inherit the extractor's errors (measured, not eliminated); BERTScore was excluded from bootstrap CIs for compute reasons; a single seed per configuration.
- **LLM-judge option** (an LLM labelling findings in generated text) was considered and dropped to avoid judge bias on top of extractor bias — the rule extractor is at least transparent and fully reproducible.
- **Compute:** a single budget-capped GPU (free tiers, then a $30 pay-per-second credit on Modal) bounds model size (3–4B), epochs (2), and seeds (1); no vision-tower adaptation; greedy decoding only.
- **Not a clinical tool:** this is a research exercise; deployment would need radiologist review, calibration, and prospective validation.
- **Perspectives:** multi-seed runs, preference-optimised decoding against clinical labels, chest-lat specific laterality checks in the extractor, and scaling to MIMIC-CXR.

## 8.5 Conclusion — to complete

On a 766-report patient-level test set of IU X-Ray, a QLoRA-adapted Qwen2.5-VL-3B (1.86% of parameters trained, ~3 h on one A10G) generates fluent chest X-ray reports — best ROUGE-L (0.255) and BERTScore (0.881) of all five systems — but is only partially clinically faithful: clinical-label macro-F1 **0.084** (95% CI 0.068–0.100), clearly above the normal-template baseline (0.038, CI 0.033–0.044) yet clearly below retrieval (0.174, CI 0.139–0.208) and the zero-shot VLM (0.152, CI 0.125–0.179). Class-balanced retraining lifted macro-F1 from exactly 0.00 (the documented collapse run) to 0.084, but at the cost of flipping the failure mode: from silent omission (455 misses, 0 false positives) to over-called normality (588/766 reports with a false-positive assertion, driven by "no acute finding" at precision 0.17) — clinically the more dangerous error. The r=32 vs r=8 ablation (0.0838 vs 0.0856, within CI noise) shows this is not a capacity limit: with a frozen vision tower and two epochs, the adapter learns the corpus output prior more readily than image-conditioned findings. The honest margins: single seed, single institution, extractor agreement measured at 84.3% (354/420 labels on 60 manually validated reports; weakest label: no acute finding at 48.3%) even though the same extractor judges all systems symmetrically. The single most important lesson about evaluating medical text generation: overlap metrics and clinical faithfulness rank systems almost oppositely here (zero-shot VLM last on BLEU-4 at 0.018, second on macro-F1 at 0.152) — evaluating medical report generation without a clinical-faithfulness measure would have inverted every conclusion in this project.

## 8.6 References

1. D. Demner-Fushman et al., 'Preparing a Collection of Radiology Examinations for Distribution and Retrieval', JAMIA, 2016 — the IU X-Ray / Open-I collection. DOI 10.1093/jamia/ocv080.
2. Z. Chen, Y. Song, T.-H. Chang, X. Wan, 'Generating Radiology Reports via Memory-driven Transformer' (R2Gen), EMNLP 2020, arXiv:2010.16056.
3. J. Irvin et al., 'CheXpert: A Large Chest Radiograph Dataset with Uncertainty Labels and Expert Comparison', AAAI 2019, arXiv:1901.07031.
4. A. E. W. Johnson et al., 'MIMIC-CXR, a De-identified Publicly Available Database of Chest Radiographs with Free-text Reports', Scientific Data, 2019. DOI 10.1038/s41597-019-0322-0.
5. S. Bai et al., 'Qwen2.5-VL Technical Report', 2025, arXiv:2502.13923.
6. A. Sellergren et al., 'MedGemma Technical Report', 2025, arXiv:2507.05201.
7. E. Hu et al., 'LoRA: Low-Rank Adaptation of Large Language Models', ICLR 2022, arXiv:2106.09685.
8. T. Dettmers, A. Pagnoni, A. Holtzman, L. Zettlemoyer, 'QLoRA: Efficient Finetuning of Quantized LLMs', NeurIPS 2023, arXiv:2305.14314.
9. T. Zhang et al., 'BERTScore: Evaluating Text Generation with BERT', ICLR 2020, arXiv:1904.09675.
10. K. Papineni et al., 'BLEU: a Method for Automatic Evaluation of Machine Translation', ACL 2002. C.-Y. Lin, 'ROUGE: A Package for Automatic Evaluation of Summaries', Text Summarization Branches Out, 2004.

*(All arXiv identifiers and DOIs above were verified against the arXiv and DOI resolvers on 2026-10-07. Reference 2's original identifier (2006.11313) was found to resolve to an unrelated paper and was corrected to 2010.16056.)*

## 8.7 Reproducibility statement

All random generators are seeded (`SEED=42`); library versions and GPU are logged in `VERSION_LOG` (Stage 1.2); the data pipeline asserts the patient-level split at run time (Stage 2.4); every system's generations are persisted as JSON and every metric is recomputed from those files, so each stage can be re-run independently in run order 1→8.

In [ ]:
# =====================================================================
# STAGE 7.5 — End-of-run packaging (runs on every platform)
# Bundles the whole persistent dir into ONE zip and, on Modal, also
# copies this notebook file (as last saved — press Ctrl+S just before
# running this cell so it carries its outputs) next to it in the
# persistent Volume. The session's outputs then survive even if the
# browser dies before anything is downloaded. Retrieve afterwards from
# the Jupyter file browser (PERSIST/artefacts_pack.zip) or, from the
# laptop:  modal volume get <volume-name> / .
# =====================================================================
import shutil as _shutil

print('\n================= END-OF-RUN PACKAGING =================')
try:
    _base = os.path.join(LOCAL, 'artefacts_pack')        # build OUTSIDE PERSIST
    _shutil.make_archive(_base, 'zip', PERSIST)          # ... so the zip
    _pack = os.path.join(PERSIST, 'artefacts_pack.zip')  # cannot contain itself
    if os.path.exists(_pack):
        os.remove(_pack)
    _shutil.move(_base + '.zip', _pack)
    print(f'pack written: {_pack}  ({os.path.getsize(_pack)/1e6:.1f} MB)')

    _nb_candidates = [os.path.join(os.getcwd(), 'medical_report_generation.ipynb'),
                      '/root/lab/mount/medical_report_generation.ipynb']
    _nb = next((p for p in _nb_candidates if os.path.exists(p)), None)
    if _nb:
        _shutil.copy2(_nb, os.path.join(PERSIST, 'medical_report_generation.ipynb'))
        print('notebook copied into PERSIST:', _nb)
    else:
        print('notebook file not found next to the kernel — copy it manually')

    print('\n---- final results table ----')
    if os.path.exists(os.path.join(PERSIST, 'main_results_table.csv')):
        from IPython.display import display
        display(pd.read_csv(os.path.join(PERSIST, 'main_results_table.csv')))
    elif os.path.exists(RESULTS_PATH):
        print(pd.read_csv(RESULTS_PATH).to_string(index=False))
    else:
        print('(no results table)')

    for _tag in ('m_r32', 'm_r8'):
        _lp = os.path.join(PERSIST, f'train_log_{_tag}.json')
        if os.path.exists(_lp):
            with open(_lp) as f:
                _lg = json.load(f)
            print(f'train_log_{_tag}:', {k: _lg[k] for k in
                  ('lora_r', 'epochs', 'hours', 'final_loss_50', 'peak_gpu_gb')})
        else:
            print(f'train_log_{_tag}: MISSING (training did not finish)')
except Exception:
    import traceback as _tb
    print('!! packaging hit an error (artefacts on disk are unaffected):')
    _tb.print_exc()
